# EXP-001 · Qué soportan de verdad los datos sobre un extent

## 1 · Para qué sirve este notebook y cómo se usa

Antes de construir ningún pipeline hay que saber qué entregan **realmente** los servicios
de ortofoto, catastro y elevación. Este notebook lo mide sobre un único rectángulo del
mapa y responde cuatro preguntas — o dice por qué no ha podido. Es el instrumento de
[EXP-001](../docs/experiments/exp-001-data-feasibility.md).

| # | Pregunta | La miden las secciones |
|---|---|---|
| 1 | ¿A qué resolución y en qué años cubre la ortofoto el extent? | 5 y 6 |
| 2 | ¿Qué productos de elevación lo cubren, y en qué forma? | 9 |
| 3 | ¿Qué indicadores de abandono se pueden calcular con eso? | 7 y 12 |
| 4 | ¿Se pueden etiquetar a mano suficientes ejemplos? ¿Ayuda el catastro? | 8, 10 y 11 |

Se ejecuta con dos comandos, desde la raíz del repositorio:

```bash
uv run --group eda python notebooks/exp-001/eda.py --selftest   # comprobaciones sin red
uv run --group eda jupyter lab notebooks/exp-001/01-eda-pnoa-catastro.ipynb
```

El experimento es **autocontenido**: todo vive en esta carpeta, datos incluidos, así que
se puede mover entera sin romper nada. Lo que escribe cae en `out/`, junto a este
notebook: `manifest.csv` —una línea por petición, con URL, estado, bytes y MD5—,
`extent.json`, `drawing.json`, `candidates.csv`, `summary.md`, y los cuerpos de cada
respuesta archivados por tipo en `capabilities/`, `vectors/`, `features/`, `epochs/` y
`chips/`. Las dos últimas, que son imágenes y se pueden volver a pedir, están ignoradas
por git; el resto se versiona, porque es la auditoría. Esa es la razón de ser del
manifiesto: **una cifra sin su línea en el manifiesto no es una medición**.

Vocabulario mínimo, para que el resto se lea sin tropiezos:

| Término | En una línea |
|---|---|
| **extent** | el rectángulo de estudio: cuatro números (mín. x, mín. y, máx. x, máx. y) |
| **CRS** | el sistema de coordenadas en que están escritos esos números (`EPSG:4326` son grados) |
| **GSD** | cuántos metros de suelo mide un píxel: 0,25 m/px es un píxel por cada 25 cm |
| **época** | una pasada del vuelo fotogramétrico, en la práctica un año de ortofoto |
| **tesela** | el recorte cuadrado fijo con el que el pipeline trocea la imagen |
| **`ABSENT:`** | el servicio no dio el dato, y la línea dice por qué. Es un resultado, no un fallo |

Los servicios se alcanzan desde tu máquina y, a propósito, **no** desde el sandbox de los
agentes: una ejecución ahí dentro reporta ausencia en todo, y eso es correcto.

**Qué mirar:** si ninguna sección llega a hablar con un servicio, toda cifra de abajo es
`ABSENT` y lo que estás midiendo es tu acceso a la red, no el dato.


## 2 · Parámetros: las únicas líneas que se editan

La celda siguiente reúne **todo** aquello de lo que depende esta ejecución, una variable
por línea con su comentario. Cambiar de extent, de proveedor o de rango de años es editar
esta celda y ninguna otra.

Dos valores están **vacíos a propósito**: `EXTENT` y `EXTENT_CRS`. Elegir el extent es el
paso 1 de EXP-001 y sigue abierto, así que aquí no hay ningún rectángulo escrito: o lo
dibujas en el mapa de la sección 3, o pegas sus cuatro números aquí a mano.

Los hosts, los nombres de capa y los atributos que se esperan del catastro **no están en
el notebook**: viven en la entrada `PROVIDERS[PROVIDER_KEY]` de `eda.py`, el único
sitio del proyecto donde se nombra un servicio. El tamaño de píxel también sale de ahí
(`default_gsd_m`), nunca de un número escrito a mano, así que cubrir otro país es añadir
una entrada en vez de tocar este notebook.

**Qué mirar:** que el panel que imprime la celda muestre el proveedor, el rango de años y
la carpeta de salidas que esperabas, y que el extent aparezca como pendiente.


In [1]:
# ═══ Parámetros · las únicas líneas que se editan ═══════════════════════════
EXTENT: tuple[float, ...] = ()  # mín. x, mín. y, máx. x, máx. y — vacío: el paso 1 de EXP-001 sigue abierto
EXTENT_CRS = ""  # el CRS en que están escritos esos cuatro números — vacío hasta elegirlo
PROVIDER_KEY = "es-pnoa"  # qué entrada de PROVIDERS (eda.py) describe los servicios
YEAR_FIRST = 2004  # primer año del rango de épocas que sondea la sección 5
YEAR_LAST = 2024  # último año de ese rango, incluido — 2024 es el último que anuncia el servicio
REQUEST_PX = 512  # ancho y alto, en píxeles, de cada imagen pedida sobre el extent completo
TILE_PX = 256  # tamaño de tesela del pipeline: fija el número de teselas y el tamaño del chip
SNAP_GRID_M = 0.0  # rejilla de ajuste en metros; 0.0 significa TILE_PX × GSD, o sea una tesela entera
BUILDING_SIDE_M = 10.0  # lado del edificio SUPUESTO, en metros: solo para contar píxeles por edificio en la sección 6
N_EXAMPLES = 6  # cuántos edificios de ejemplo llevan tira de chips en la sección 10
LADDER_YEARS_SHOWN = 3  # cuántos de los últimos años con escalera dibuja la sección 6.2
OUT_DIR_NAME = "out"  # dónde va todo, al lado de este notebook: el experimento se mueve entero
# ═══ Debajo de esta línea no hay parámetros ════════════════════════════════

from pathlib import Path

from IPython.display import Markdown, display

import eda

PROVIDER = eda.PROVIDERS[PROVIDER_KEY]
GSD_M = PROVIDER.default_gsd_m  # tamaño de píxel: dato del proveedor, nunca un literal
SNAP_M = SNAP_GRID_M or TILE_PX * GSD_M  # la rejilla efectiva a la que se ajusta el extent
YEARS = range(YEAR_FIRST, YEAR_LAST + 1)
HERE = Path(eda.__file__).resolve().parent  # la carpeta del experimento, no el cwd
OUT_DIR = eda.prepare_out(HERE / OUT_DIR_NAME)  # crea out/ y una subcarpeta por tipo


def panel(title: str, rows: list[tuple[str, str]], note: str = "") -> None:
    """Muestra pares etiqueta/valor como una tabla, con una nota opcional debajo.

    Es la forma en que todas las secciones presentan sus resultados: una tabla
    legible en lugar de una ráfaga de `print`.
    """
    table = "\n".join(f"| {label} | {value} |" for label, value in rows)
    tail = f"\n\n{note}" if note else ""
    display(Markdown(f"**{title}**\n\n| | |\n|---|---|\n{table}{tail}"))


extent_row = (
    f"`{tuple(EXTENT)}` en `{EXTENT_CRS or 'CRS sin indicar'}`"
    if EXTENT
    else "_vacío — se elige en la sección 3_"
)
panel(
    "Configuración de esta ejecución",
    [
        ("📍 Proveedor", f"{PROVIDER.label} (`{PROVIDER_KEY}`)"),
        ("🗺️ CRS nativo", f"`{PROVIDER.native_crs}` a {GSD_M} m/px"),
        ("📅 Épocas a sondear", f"{YEAR_FIRST}–{YEAR_LAST} ({len(YEARS)} años)"),
        ("🖼️ Tamaño de petición", f"{REQUEST_PX} px sobre el extent · tesela {TILE_PX} px"),
        ("📐 Rejilla de ajuste", f"{SNAP_M:g} m (= {TILE_PX} px × {GSD_M} m)"),
        ("🏠 Edificios de ejemplo", f"{N_EXAMPLES}"),
        ("💾 Salidas", f"carpeta `{OUT_DIR.relative_to(HERE)}` (imágenes ignoradas por git)"),
        ("⏳ Extent", extent_row),
    ],
    note="Esta celda no pide nada a ningún servicio: solo configura y crea la carpeta de salidas.",
)


**Configuración de esta ejecución**

| | |
|---|---|
| 📍 Proveedor | Spain: PNOA orthophotos and cadastral building footprints (`es-pnoa`) |
| 🗺️ CRS nativo | `EPSG:25830` a 0.25 m/px |
| 📅 Épocas a sondear | 2004–2024 (21 años) |
| 🖼️ Tamaño de petición | 512 px sobre el extent · tesela 256 px |
| 📐 Rejilla de ajuste | 64 m (= 256 px × 0.25 m) |
| 🏠 Edificios de ejemplo | 6 |
| 💾 Salidas | carpeta `out` (imágenes ignoradas por git) |
| ⏳ Extent | _vacío — se elige en la sección 3_ |

Esta celda no pide nada a ningún servicio: solo configura y crea la carpeta de salidas.

### 2.1 · Utilidades compartidas: cómo se anota cada veredicto

Las secciones de abajo **no lanzan excepciones** cuando un servicio no contesta: anotan la
ausencia con su motivo y siguen adelante. Esta celda define las tres funciones que usan
todas, y por eso va antes que cualquier medición:

- `record(sección, línea)` — guarda e imprime la única línea de veredicto de una sección;
  la sección 12 las recoge todas para el resumen final.
- `absent(sección, motivo)` — lo mismo, pero la línea empieza por `ABSENT: ` y dice **por
  qué** no hubo dato.
- `attempt(qué, llamada)` — envuelve una petición que puede fallar *antes* de salir de esta
  máquina (un almacén de certificados sin CA, un proxy mal configurado). Devuelve el mismo
  tipo de valor que una petición normal, con el motivo dentro de `error`.

Nada se oculta: el motivo viaja siempre en el resultado. Eso es lo que permite leer el
notebook de principio a fin como un mapa de lo que hay y lo que falta, en vez de encontrar
un traceback a mitad y no saber cuánto se llegó a medir.

**Qué mirar:** la línea de confirmación al final de la celda. A partir de aquí, cualquier
`ABSENT:` que aparezca abajo viene con su motivo pegado.


In [2]:
# --- Cómo se registra lo que mide cada sección -----------------------------
%matplotlib inline

from collections.abc import Callable

FINDINGS: dict[str, str] = {}  # sección -> su línea de veredicto, recogidas por la sección 12
MEASURED: dict[str, object] = {}  # sección -> las cifras que la sección 12 puede llamar medidas


def record(section: str, line: str) -> None:
    """Guarda la línea de veredicto de una sección y la imprime."""
    FINDINGS[section] = line
    print(line)


def absent(section: str, reason: str) -> None:
    """Anota que una sección no tuvo nada que medir, y por qué."""
    record(section, "ABSENT: " + reason)


def attempt(what: str, call: Callable[[], eda.Fetch]) -> eda.Fetch:
    """Lanza una petición que puede fallar antes incluso de salir de esta máquina.

    `eda.fetch` ya convierte un fallo de red en un valor; esto hace lo mismo con
    un fallo anterior a la red, para que quien llame lo reporte con sus palabras.
    """
    try:
        return call()
    except OSError as exc:
        return eda.Fetch(
            url=f"no enviada: {what}",
            status=0,
            nbytes=0,
            md5="",
            elapsed_s=0.0,
            path=None,
            error=f"{type(exc).__name__}: {exc}",
        )


print("utilidades listas · veredictos: record() / absent() · peticiones: attempt()")
print(f"manifiesto de la ejecucion: {eda.manifest_path(OUT_DIR).relative_to(HERE)}")


utilidades listas · veredictos: record() / absent() · peticiones: attempt()
manifiesto de la ejecucion: out/manifest.csv


## 3 · Elegir el extent dibujándolo en el mapa

El extent es la única entrada que no se deduce de nada, así que se elige a mano. El mapa
abre sobre **imagen de satélite** (basemap Esri) para que nunca dibujes a ciegas, y encima
lleva la **ortofoto del proveedor** como capa conmutable desde el control de capas: esa es
la imagen que se va a medir de verdad, y si sale en negro o con teselas de error, el nombre
de capa de la entrada del proveedor está mal — lo confirma la sección 4. Solo está activa
la herramienta de **rectángulo**; un extent es un rectángulo o no es nada.

Tres pasos, en este orden:

1. **Dibuja un rectángulo** (o reajústalo: la edición está activada). Al soltar el ratón
   verás sus esquinas SO/NE impresas debajo del mapa, y el rectángulo queda guardado en
   `drawing.json` en ese mismo instante.
2. **Pulsa «Usar este rectángulo».** No es decorativo: es la prueba de que el kernel está
   recibiendo lo que haces. Si el texto de al lado cambia, el puente navegador↔kernel
   funciona; si no cambia, está muerto y lo que toca es rellenar `EXTENT` y `EXTENT_CRS` en
   la sección 2.
3. **Ejecuta la celda siguiente**, que lo resuelve y lo mide.

Un par de cientos de metros de lado es el orden de magnitud correcto: pequeño para poder
revisar a mano todos sus edificios en la sección 10, grande para que haya unas decenas.
Mientras dibujas no se pide nada a ningún servicio: las teselas las carga tu navegador.
*Run All* **no puede** funcionar aquí, porque entre estas dos celdas hace falta una persona.

**Qué mirar:** que el texto junto al botón pase a describir tu rectángulo. Eso, y solo eso,
confirma que el valor ha llegado al kernel.


In [3]:
# --- El mapa, el rectángulo y el botón que confirma que llegó al kernel ----
import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import (
    DrawControl,
    LayersControl,
    Map,
    MeasureControl,
    ScaleControl,
    WMSLayer,
    basemap_to_tiles,
    basemaps,
)

DRAW_CRS = "EPSG:4326"  # un mapa web devuelve los rectángulos en longitud y latitud
SELECTION: dict[str, object] = {}  # el último rectángulo que ha llegado al kernel
DRAW_EVENTS: list[str] = []  # lo que el kernel ha recibido del mapa: el diagnóstico

view_lat, view_lon, view_zoom = PROVIDER.default_view 
extent_map = Map(
    basemap=basemap_to_tiles(basemaps.Esri.WorldImagery),  # satélite: nunca se dibuja a ciegas
    center=(view_lat, view_lon),
    zoom=view_zoom,
    scroll_wheel_zoom=True,
)

# La ortofoto del proveedor encima del satélite. El endpoint y el nombre de capa salen de
# la entrada PROVIDERS, y las dos búsquedas tienen valor por defecto en vez de reventar:
# un nombre de capa que el servicio no sirve degrada a teselas de error en el navegador.
# Las teselas las pide el navegador, así que esta celda no toca la red.
ortho_service, ortho_layer = PROVIDER.layers.get("ortho", (PROVIDER.epoch_service, ""))
extent_map.add(
    WMSLayer(
        url=PROVIDER.endpoints.get(ortho_service, ""),
        layers=ortho_layer,
        format="image/jpeg",  # una ortofoto es fotográfica y opaca
        transparent=False,
        attribution=PROVIDER.label,
        name=f"ortofoto: {ortho_layer or 'sin capa en la entrada'} (sin confirmar)",
    )
)

draw_control = DrawControl(
    rectangle={"shapeOptions": {"color": "#d95f02", "weight": 2, "fillOpacity": 0.1}},
    polyline={},  # el resto de herramientas, desactivadas: hay una sola forma de usar esto
    polygon={},
    circle={},
    circlemarker={},
    marker={},
    edit=True,  # se puede reajustar el rectángulo arrastrando sus vértices
    remove=True,  # la papelera, para tirar uno del tamaño equivocado
)
coord_label = widgets.Label("Sin rectángulo todavía. Dibuja uno en el mapa.")
use_button = widgets.Button(description="Usar este rectángulo", button_style="success")


def rectangle_bbox(feature: object) -> tuple[float, ...]:
    """Los cuatro números de un rectángulo GeoJSON, o () si eso no es un polígono."""
    geometry = feature.get("geometry") if isinstance(feature, dict) else None
    geometry = geometry or {}
    if geometry.get("type") != "Polygon":
        return ()
    ring = [(float(x), float(y)) for x, y in geometry["coordinates"][0]]
    return (
        min(p[0] for p in ring),
        min(p[1] for p in ring),
        max(p[0] for p in ring),
        max(p[1] for p in ring),
    )


def describe(bbox: tuple[float, ...]) -> str:
    """El rectángulo en las dos esquinas que una persona puede comprobar a ojo."""
    report = eda.extent_report(bbox, DRAW_CRS, gsd_m=GSD_M, tile_px=TILE_PX)
    return (
        f"SO ({bbox[1]:.6f}, {bbox[0]:.6f}) · NE ({bbox[3]:.6f}, {bbox[2]:.6f}) · "
        f"{report['width_m']:.0f} × {report['height_m']:.0f} m · {report['area_km2']:.4f} km²"
    )


def handle_draw(target: object, action: str, geo_json: object, **kwargs: object) -> None:
    """Corre en el kernel al dibujar, editar o borrar. Imprime y guarda en el acto.

    Guardar aquí, y no al resolver, es lo que hace que el rectángulo sobreviva a un
    reinicio del kernel o a reejecutar esta celda. Se imprime en la salida de la celda
    (no en un widget `Output`) porque un fallo dentro de un callback que escribe en un
    widget desaparece sin rastro, y aquí hace falta verlo.
    """
    DRAW_EVENTS.append(action)
    if action == "deleted":
        SELECTION.clear()
        gone = eda.forget_drawing(OUT_DIR)
        coord_label.value = "Rectángulo borrado. Dibuja otro."
        print("rectángulo borrado" + (" · drawing.json eliminado" if gone else ""))
        return
    bbox = rectangle_bbox(geo_json)
    if not bbox:
        print(f"el kernel recibió '{action}', pero sin rectángulo dentro: {geo_json!r}")
        return
    SELECTION["bbox"] = bbox
    SELECTION["crs"] = DRAW_CRS
    saved = eda.save_drawing(bbox, DRAW_CRS, OUT_DIR)
    coord_label.value = describe(bbox)
    print(f"rectángulo {action} · {describe(bbox)}")
    print(f"guardado en {saved.relative_to(HERE)} — pulsa el botón y ejecuta la celda siguiente")


def use_selection(button: object) -> None:
    """Confirma el rectángulo: reescribe `drawing.json` y lo dice en la etiqueta."""
    if not SELECTION.get("bbox"):
        coord_label.value = (
            "Ningún rectángulo en el kernel. Si has dibujado uno y esto no cambia, "
            "el puente widget↔kernel no entrega: usa EXTENT en la sección 2."
        )
        return
    bbox = tuple(SELECTION["bbox"])
    saved = eda.save_drawing(bbox, str(SELECTION["crs"]), OUT_DIR)
    coord_label.value = f"✅ en uso · {describe(bbox)} · guardado en {saved.name}"
    print(f"confirmado: {eda.extent_env_lines(bbox, DRAW_CRS)}")


draw_control.on_draw(handle_draw)
use_button.on_click(use_selection)
extent_map.add(draw_control)
extent_map.add(
    MeasureControl(
        position="bottomleft",
        primary_length_unit="meters",
        secondary_length_unit="kilometers",
        primary_area_unit="sqmeters",
    )
)
extent_map.add(ScaleControl(position="bottomright"))
extent_map.add(LayersControl(position="topright"))  # conmuta ortofoto / satélite

display(extent_map)
display(use_button)
display(coord_label)


Map(center=[40.4, -3.7], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

Button(button_style='success', description='Usar este rectángulo', style=ButtonStyle())

Label(value='Sin rectángulo todavía. Dibuja uno en el mapa.')

### 3.1 · Resolver, medir y registrar el extent

Esta celda decide qué extent usa el resto del notebook y **dice de dónde lo ha sacado**,
porque una cifra medida sobre el extent equivocado es peor que no tener cifra. Busca en
este orden y se queda con el primero que encuentra:

1. el rectángulo que tiene el kernel, el que llegó por el evento del mapa o por el botón;
2. lo que guarda el propio control de dibujo (`draw.data`, `last_draw`), por si el evento
   no llegó pero su estado sí se sincronizó;
3. `EXTENT` y `EXTENT_CRS` de la sección 2, si los has rellenado a mano;
4. `drawing.json`, un rectángulo dibujado antes — **esto es lo que hace que reiniciar el
   kernel no te obligue a volver a dibujar**;
5. nada: imprime `ABSENT: ` con el motivo y un diagnóstico de qué recibió el kernel, las
   secciones siguientes reportan ausencia, y el notebook llega al final sin romperse.

Un rectángulo dibujado en un mapa web llega en longitud y latitud, que no es un sistema en
el que se puedan medir metros, así que se reproyecta al CRS nativo del proveedor como
**envolvente de las cuatro esquinas**: reproyectar solo dos esquinas opuestas perdería
terreno por dos lados. `skew_m` es el precio de haber dibujado en grados —cuánto se separan
en X las dos esquinas occidentales al proyectar— y la envolvente es lo que lo paga; el
tamaño geodésico va al lado como comprobación de cordura.

Luego se ajusta hacia fuera a teselas enteras, y ese extent ajustado es el que se escribe
en `extent.json` y el que se pega en EXP-001. Redibuja y reejecuta las veces que quieras:
sobreescribe en vez de acumular, y **nunca borra** un extent ya registrado.

**Qué mirar:** la fila marcada con ✅ en la tabla de procedencia, y `exact_fit` en el
extent ajustado — `True` significa que la rejilla de teselas lo divide sin resto.


In [4]:
# --- De dónde sale el extent, en orden, y qué mide ------------------------
candidates: list[tuple[str, tuple[float, ...], str]] = []

if SELECTION.get("bbox"):  # 1 · el kernel tiene el rectángulo
    candidates.append(
        ("el rectángulo que tiene el kernel", tuple(SELECTION["bbox"]), str(SELECTION["crs"]))
    )

for feature in reversed(list(draw_control.data)):  # 2 · el estado del control de dibujo
    from_data = rectangle_bbox(feature)
    if from_data:
        candidates.append(("el control de dibujo (data)", from_data, DRAW_CRS))
        break

if draw_control.last_action not in {"", "deleted"}:  # 2b · dibujado y no borrado después
    from_last = rectangle_bbox(draw_control.last_draw)
    if from_last:
        candidates.append(("el control de dibujo (last_draw)", from_last, DRAW_CRS))

if len(EXTENT) == 4 and EXTENT_CRS:  # 3 · escrito a mano en la sección 2
    candidates.append(
        ("el parámetro EXTENT de la sección 2", tuple(float(v) for v in EXTENT), EXTENT_CRS)
    )

previous = eda.load_drawing(OUT_DIR)  # 4 · un dibujo de antes del reinicio
if previous is not None:
    candidates.append((f"drawing.json, dibujado el {previous[2]}", previous[0], previous[1]))

EXTENT_BBOX: tuple[float, ...] = ()
EXTENT_IN = ""

if not candidates:
    panel(
        "⏳ Sin extent todavía",
        [
            ("Eventos recibidos del mapa", f"{DRAW_EVENTS or 'ninguno'}"),
            ("Rectángulos en el control", f"{len(draw_control.data)}"),
            ("Última acción del control", f"`{draw_control.last_action or 'ninguna'}`"),
            ("drawing.json", "no existe"),
            ("EXTENT en la sección 2", "vacío"),
        ],
        note=(
            "Si dibujaste un rectángulo y aquí no hay ningún evento, el puente "
            "navegador↔kernel no está entregando nada: rellena `EXTENT` y `EXTENT_CRS` "
            "en la sección 2 y reejecuta esta celda."
        ),
    )
    absent(
        "3 extent",
        "ningún extent seleccionado — dibuja un rectángulo arriba, o rellena EXTENT y "
        "EXTENT_CRS en la sección 2",
    )
    stale = OUT_DIR / "extent.json"
    if stale.is_file():
        print(f"nota: {stale.name} de una ejecución anterior sigue en disco, y no se usa aquí")
else:
    source, bbox, bbox_crs = candidates[0]
    panel(
        "De dónde sale el extent",
        [
            ("✅ usado" if n == 0 else "· disponible", name)
            for n, (name, _, _) in enumerate(candidates)
        ],
        note="Se usa el primero de la lista; los demás quedan como respaldo.",
    )
    geographic = eda.to_projected(bbox, bbox_crs, DRAW_CRS)
    native = eda.to_projected(bbox, bbox_crs, PROVIDER.native_crs)
    as_chosen = eda.extent_report(bbox, bbox_crs, gsd_m=GSD_M, tile_px=TILE_PX)
    EXTENT_BBOX = eda.snap_outward(native, SNAP_M)
    EXTENT_IN = PROVIDER.native_crs
    snapped = eda.extent_report(EXTENT_BBOX, EXTENT_IN, gsd_m=GSD_M, tile_px=TILE_PX)
    saved = eda.save_extent(EXTENT_BBOX, EXTENT_IN, OUT_DIR, gsd_m=GSD_M, tile_px=TILE_PX)
    panel(
        f"El extent elegido, medido · {source}",
        [
            ("🧭 Tal cual, en grados", f"`{tuple(round(v, 6) for v in geographic)}`"),
            ("📦 Envolvente en el CRS nativo", f"`{tuple(round(v, 2) for v in native)}`"),
            ("📏 Tamaño", f"{as_chosen['width_m']:.1f} × {as_chosen['height_m']:.1f} m"),
            ("🧮 Área", f"{as_chosen['area_km2']:.4f} km²"),
            (
                "🌍 Tamaño geodésico",
                f"{as_chosen['geodesic_width_m']:.1f} × {as_chosen['geodesic_height_m']:.1f} m"
                " (comprobación de cordura)",
            ),
            (
                "↘️ skew_m",
                f"{as_chosen['skew_m']:.2f} m — lo que se separan en X las dos esquinas"
                " occidentales al proyectar; la envolvente lo absorbe, no se pierde terreno",
            ),
            (
                "🧱 Ajustado a la rejilla",
                f"`{tuple(round(v, 2) for v in EXTENT_BBOX)}` ({SNAP_M:g} m)",
            ),
            ("🖼️ Píxeles", f"{snapped['width_px']} × {snapped['height_px']} px a {GSD_M} m/px"),
            (
                "🧩 Teselas",
                f"{snapped['tiles'][0]} × {snapped['tiles'][1]} de {TILE_PX} px ·"
                f" exact_fit={snapped['exact_fit']}",
            ),
            ("💾 Registrado en", f"`{saved.relative_to(HERE)}`"),
        ],
    )
    print("pegar en EXP-001 -> Configuration:")
    print(eda.extent_env_lines(EXTENT_BBOX, EXTENT_IN))
    MEASURED["extent"] = {"bbox": list(EXTENT_BBOX), "crs": EXTENT_IN, "report": snapped}
    record(
        "3 extent",
        f"extent desde {source}: {snapped['width_px']}x{snapped['height_px']} px en "
        f"{EXTENT_IN}, {snapped['tiles'][0]}x{snapped['tiles'][1]} teselas, "
        f"guardado como {saved.name}",
    )


**De dónde sale el extent**

| | |
|---|---|
| ✅ usado | drawing.json, dibujado el 2026-11-04T19:18:23+00:00 |

Se usa el primero de la lista; los demás quedan como respaldo.

**El extent elegido, medido · drawing.json, dibujado el 2026-11-04T19:18:23+00:00**

| | |
|---|---|
| 🧭 Tal cual, en grados | `(-3.364126, 40.497182, -3.348985, 40.504393)` |
| 📦 Envolvente en el CRS nativo | `(469144.88, 4483000.42, 470431.06, 4483806.03)` |
| 📏 Tamaño | 1286.2 × 805.6 m |
| 🧮 Área | 1.0362 km² |
| 🌍 Tamaño geodésico | 1283.5 × 800.7 m (comprobación de cordura) |
| ↘️ skew_m | 3.30 m — lo que se separan en X las dos esquinas occidentales al proyectar; la envolvente lo absorbe, no se pierde terreno |
| 🧱 Ajustado a la rejilla | `(469120.0, 4482944.0, 470464.0, 4483840.0)` (64 m) |
| 🖼️ Píxeles | 5376 × 3584 px a 0.25 m/px |
| 🧩 Teselas | 21 × 14 de 256 px · exact_fit=True |
| 💾 Registrado en | `out/extent.json` |

pegar en EXP-001 -> Configuration:
GEO_DECAY_BBOX=469120.0,4482944.0,470464.0,4483840.0
GEO_DECAY_BBOX_CRS=EPSG:25830
extent desde drawing.json, dibujado el 2026-11-04T19:18:23+00:00: 5376x3584 px en EPSG:25830, 21x14 teselas, guardado como extent.json


## 4 · Capacidades: confirmar o contradecir la entrada del proveedor

Antes de medir nada **a través** de la entrada del proveedor hay que ver si lo que afirma
existe. Esta celda pide a cada servicio su documento de capacidades —la lista de lo que
sirve— y lo contrasta afirmación por afirmación: cada CRS, cada nombre de capa, el patrón
de épocas, la stored query del WFS y los atributos esperados del catastro.

Cada fila sale con su veredicto: ✅ `ok` si el servicio lo anuncia, ⚠️ `unverified` si
desde aquí no se puede saber, ❌ `MISMATCH` si dice otra cosa — y entonces la fila trae los
nombres más parecidos que sí anuncia. **Un valor equivocado se reporta, no se arregla
aquí.** Es además la sección más barata de ejecutar, y la que decide si merece la pena
sondear 22 épocas a través de esta entrada.

**Qué mirar:** la columna `veredicto`. Un ❌ en la capa de ortofoto o en el patrón de
épocas invalida todo lo que venga después; uno en el WFS afecta solo a la sección 8.


In [5]:
# --- Sección 4: lo que cada servicio anuncia, contra lo que la entrada afirma ---
import difflib

import pandas as pd

CAPS: dict[str, eda.Capabilities] = {}  # servicio -> lo que anuncia
STORED: tuple[str, ...] = ()  # las stored queries que ofrece el WFS

FEATURE_SERVICES = eda.wfs_services(PROVIDER)  # los que sirven features, no mapas

for service in PROVIDER.endpoints:
    if service in FEATURE_SERVICES:
        got = attempt(service, lambda s=service: eda.wfs_capabilities(PROVIDER, s, OUT_DIR))
    else:
        got = attempt(service, lambda s=service: eda.wms_capabilities(PROVIDER, s, OUT_DIR))
    caps = eda.read_capabilities(got.path)
    CAPS[service] = caps
    reason = got.error or caps.exception
    if reason:
        absent(f"4 {service}", f"{service} — {reason}")
    else:
        record(
            f"4 {service}",
            f"{service}: {got.nbytes} bytes, md5 {got.md5[:8]}, "
            f"{len(caps.crs)} CRS, {len(caps.layers)} capas anunciadas",
        )
    if service in FEATURE_SERVICES:
        listed = attempt(
            f"{service} ListStoredQueries",
            lambda s=service: eda.wfs_stored_queries(PROVIDER, s, OUT_DIR),
        )
        STORED = STORED + eda.read_capabilities(listed.path).stored_queries
        if not STORED:
            absent(
                f"4 {service} stored queries",
                f"{service} ListStoredQueries — "
                f"{listed.error or 'el servicio no anuncia ninguna stored query'}",
            )

ADVERTISED = tuple(dict.fromkeys(code for caps in CAPS.values() for code in caps.crs))


def judge(claim: str, advertised: tuple[str, ...], asked: bool) -> tuple[str, str]:
    """Veredicto de una afirmación de la entrada contra lo que el servicio anuncia."""
    if not asked:
        return "⚠️ unverified", "el servicio no contestó, así que no se pudo comprobar"
    if claim in advertised:
        return "✅ ok", "anunciado"
    near = difflib.get_close_matches(claim, advertised, n=3, cutoff=0.3)
    tail = f"; más cercanos: {', '.join(near)}" if near else ""
    return "❌ MISMATCH", "no anunciado" + tail


def claim_row(what: str, claim: str, verdict: tuple[str, str]) -> dict[str, str]:
    """Una fila de la tabla: qué se afirma, quién lo afirma y qué contesta el servicio."""
    return {
        "qué": what,
        "la entrada afirma": claim,
        "veredicto": verdict[0],
        "el servicio dice": verdict[1],
    }


claims = [
    claim_row("CRS anunciado", code, judge(code, ADVERTISED, bool(ADVERTISED)))
    for code in PROVIDER.advertised_crs
]

for logical, (service, name) in PROVIDER.layers.items():
    served = CAPS.get(service, eda.read_capabilities(None)).layers
    claims.append(
        claim_row(f"capa '{logical}' en {service}", name, judge(name, served, bool(served)))
    )

for logical, (service, name) in PROVIDER.wfs_types.items():
    served = CAPS.get(service, eda.read_capabilities(None)).layers
    claims.append(
        claim_row(f"tipo '{logical}' en {service}", name, judge(name, served, bool(served)))
    )

epoch_layers = CAPS.get(PROVIDER.epoch_service, eda.read_capabilities(None)).layers
expected = [PROVIDER.epoch_layer_pattern.format(year=year) for year in YEARS]
present = [name for name in expected if name in epoch_layers]
# El patrón es correcto si el servicio anuncia al menos un año con él; lo que importa
# medir es *qué* años, porque ese es el rango real que puede sondear la sección 5.
served_years = [
    year for year in YEARS if PROVIDER.epoch_layer_pattern.format(year=year) in epoch_layers
]
if not epoch_layers:
    epoch_verdict = ("⚠️ unverified", "el servicio de épocas no contestó")
elif not served_years:
    epoch_verdict = ("❌ MISMATCH", "ningún año del rango está anunciado con este patrón")
else:
    detail = f"{len(served_years)} de {len(expected)}: {served_years[0]}–{served_years[-1]}"
    gap = [year for year in YEARS if year not in served_years]
    if gap:
        shown = ", ".join(str(year) for year in gap[:6]) + (" …" if len(gap) > 6 else "")
        detail += f"; sin anunciar: {shown}"
    epoch_verdict = ("✅ ok", detail)
claims.append(
    claim_row(
        f"patrón de épocas en {PROVIDER.epoch_service}",
        f"{PROVIDER.epoch_layer_pattern} en {len(YEARS)} años",
        epoch_verdict,
    )
)

for key, value in PROVIDER.wfs_params.items():
    if "storedquer" in key.lower():  # el nombre del parámetro es cosa de cada servicio
        claims.append(
            claim_row(f"stored query ({key})", value, judge(value, STORED, bool(STORED)))
        )

for logical, name in PROVIDER.attributes.items():
    claims.append(
        claim_row(
            f"atributo '{logical}'",
            name,
            ("⚠️ unverified", "llega con las features, no con las capacidades: sección 8"),
        )
    )

if EXTENT_IN:
    claims.append(
        claim_row("CRS de las peticiones", EXTENT_IN, judge(EXTENT_IN, ADVERTISED, bool(ADVERTISED)))
    )
else:
    absent("4 CRS de las peticiones", "sin extent no hay CRS de peticiones que comprobar")

display(pd.DataFrame(claims))
mismatches = [entry for entry in claims if "MISMATCH" in entry["veredicto"]]
answered = [name for name, caps in CAPS.items() if caps.crs or caps.layers]
panel(
    "Resumen de la sección 4",
    [
        ("📄 Servicios que contestaron", f"{len(answered)} de {len(PROVIDER.endpoints)}"),
        ("🗺️ CRS anunciados (unión)", f"{len(ADVERTISED)}"),
        ("🧱 Capas anunciadas", f"{sum(len(caps.layers) for caps in CAPS.values())}"),
        ("🔎 Stored queries", f"{len(STORED)}"),
        ("❌ Contradicciones", f"{len(mismatches)}"),
    ],
    note="Lo contradicho se corrige en la entrada del proveedor de `eda.py`, no en el notebook.",
)
MEASURED["capabilities"] = {
    "advertised_crs": list(ADVERTISED),
    "stored_queries": list(STORED),
    "claims": claims,
}
record(
    "4 capacidades",
    f"capacidades: contestaron {len(answered)}/{len(PROVIDER.endpoints)} servicios, "
    f"{len(ADVERTISED)} CRS y {sum(len(caps.layers) for caps in CAPS.values())} capas "
    f"anunciadas, {len(mismatches)} contradicciones contra la entrada",
)


ortho: 27902 bytes, md5 cb3fcd0f, 20 CRS, 3 capas anunciadas
ortho_history: 113091 bytes, md5 40b41018, 18 CRS, 37 capas anunciadas
cadastre_wms: 28799 bytes, md5 745793a5, 17 CRS, 13 capas anunciadas
buildings_wfs: 16182 bytes, md5 36610fcf, 11 CRS, 3 capas anunciadas
addresses_wfs: 13222 bytes, md5 740c51e1, 11 CRS, 1 capas anunciadas
parcels_wfs: 14510 bytes, md5 22a2693c, 14 CRS, 2 capas anunciadas


,qué,la entrada afirma,veredicto,el servicio dice
0,CRS anunciado,EPSG:25829,✅ ok,anunciado
1,CRS anunciado,EPSG:25830,✅ ok,anunciado
2,CRS anunciado,EPSG:25831,✅ ok,anunciado
3,CRS anunciado,EPSG:4326,✅ ok,anunciado
4,capa 'ortho' en ortho,OI.OrthoimageCoverage,✅ ok,anunciado
5,capa 'buildings' en cadastre_wms,CONSTRU,✅ ok,anunciado
6,tipo 'buildings' en buildings_wfs,bu:Building,✅ ok,anunciado
7,tipo 'building_parts' en buildings_wfs,bu:BuildingPart,✅ ok,anunciado
8,tipo 'other_constructions' en buildings_wfs,bu:OtherConstruction,✅ ok,anunciado
9,tipo 'addresses' en addresses_wfs,ad:Address,✅ ok,anunciado


**Resumen de la sección 4**

| | |
|---|---|
| 📄 Servicios que contestaron | 6 de 6 |
| 🗺️ CRS anunciados (unión) | 29 |
| 🧱 Capas anunciadas | 59 |
| 🔎 Stored queries | 15 |
| ❌ Contradicciones | 0 |

Lo contradicho se corrige en la entrada del proveedor de `eda.py`, no en el notebook.

capacidades: contestaron 6/6 servicios, 29 CRS y 59 capas anunciadas, 0 contradicciones contra la entrada


## 5 · Épocas: qué años contestan y qué resolución declara cada uno

Esta sección pide una imagen pequeña por cada año del rango y anota, fila por año, lo que
contesta el servicio: eso se mide pidiendo, no leyendo el catálogo.

La resolución es otra pregunta y va **siempre con su fuente**. No hay campo de escala: se
declara en prosa, en el `Abstract` de cada capa de año, y **por región**, y se leen todas
con sus palabras. Un año sin declaración sale como `no declarada`, y **nunca hereda** el
número del vecino, de la capa padre ni el `default_gsd_m` de la entrada. Aparte va la
resolución **pedida** al sondear: lo que pedimos, no lo que voló el sensor.

Ninguna es una medición: medir exige pedir el mismo bbox a más píxeles hasta que el detalle
deje de crecer. Las capacidades tampoco dicen en qué región cae el extent; eso lo contesta
su capa de cobertura por `GetFeatureInfo`, en otro paso.

**Qué mirar:** la columna `resolución declarada` junto a su `fuente`: un número sin fuente
invalida la sección.

In [6]:
# --- Sección 5: qué años contestan sobre el extent, y qué declara cada uno ---
import pandas as pd

EPOCH_CAPS = CAPS.get(PROVIDER.epoch_service, eda.read_capabilities(None))
NOT_ASKED = "no se pidió: la sección 3 no resolvió ningún extent"
EPOCH_ROWS: list[dict[str, object]] = []  # una fila por año, como contestó el servicio
DECLARED: dict[int, tuple[eda.DeclaredResolution, ...]] = {}  # año -> lo declarado

if EXTENT_BBOX:
    EPOCH_ROWS = eda.epoch_availability(
        PROVIDER, EXTENT_BBOX, EXTENT_IN, YEARS, OUT_DIR
    )
    ground = eda.extent_report(EXTENT_BBOX, EXTENT_IN, gsd_m=GSD_M, tile_px=TILE_PX)
    # Lo que PEDIMOS: metros de suelo por píxel de la imagen de sondeo. No es la
    # resolución del vuelo, y el tamaño de petición lo fija eda, no esta celda.
    asked = (
        f"{float(ground['width_m']) / eda.PROBE_SIZE[0]:.2f} × "
        f"{float(ground['height_m']) / eda.PROBE_SIZE[1]:.2f} m/px"
        f" ({eda.PROBE_SIZE[0]}×{eda.PROBE_SIZE[1]} px sobre el extent completo)"
    )
else:
    asked = NOT_ASKED
probed = {int(row["year"]): row for row in EPOCH_ROWS}

epoch_table: list[dict[str, object]] = []
for year in YEARS:
    layer = PROVIDER.epoch_layer_pattern.format(year=year)
    advertised = layer in EPOCH_CAPS.layers
    found = eda.declared_resolutions(EPOCH_CAPS.abstracts.get(layer, ""))
    DECLARED[year] = found
    if found:  # tal cual lo declara el servicio: una cifra por región nombrada
        declared = " · ".join(f"{d.gsd_m:g} m" for d in found)
        origin = f"`Abstract` de la capa, {len(found)} declaración(es)"
    elif EPOCH_CAPS.exception:  # catálogo no leído: no se hereda nada de nadie
        declared, origin = "no declarada", f"catálogo: {EPOCH_CAPS.exception}"
    elif not advertised:
        declared, origin = "no declarada", "la capa no está anunciada en el catálogo"
    else:
        declared, origin = "no declarada", "su `Abstract` no declara resolución"
    got = probed.get(year)
    verdict = str(got["verdict"]) if got else "unknown"
    evidence = str(got["evidence"]) if got else NOT_ASKED
    epoch_table.append(
        {
            "año": year,
            "anunciada": "sí" if advertised else "no",
            "veredicto": verdict,
            "evidencia": evidence,
            "blanco": got["blank"] if got else None,
            "estado": int(got["status"]) if got else 0,
            "bytes": int(got["nbytes"]) if got else 0,
            "md5": str(got["md5"])[:8] if got else "",
            "resolución declarada": declared,
            "fuente de la resolución": origin,
        }
    )
    if verdict != "present":  # una línea por año, con su motivo completo
        print(
            f"ABSENT: {year} — sin imagen ({verdict}: {evidence});"
            f" resolución {declared} ({origin})"
        )

display(pd.DataFrame(epoch_table))

quotes = [
    (f"{year} · {d.gsd_m:g} m", f"«{d.text}»")
    for year, found in DECLARED.items()
    for d in found
]
if quotes:
    panel(
        "Resolución declarada, con las palabras del servicio",
        quotes,
        note=(
            "Cada fila es una declaración, no una medición, y la cita es la región tal "
            "como la nombra el servicio. Las capacidades no dicen en qué región cae "
            "este extent: lo contesta su capa de cobertura por `GetFeatureInfo`."
        ),
    )

present_years = [
    int(row["año"]) for row in epoch_table if row["veredicto"] == "present"
]
declaring = [year for year, found in DECLARED.items() if found]
distinct = sorted({d.gsd_m for found in DECLARED.values() for d in found})
panel(
    "Resumen de la sección 5",
    [
        ("📅 Años del rango", f"{len(epoch_table)} ({YEAR_FIRST}–{YEAR_LAST})"),
        (
            "🧾 Anunciados por el servicio",
            f"{sum(1 for row in epoch_table if row['anunciada'] == 'sí')}",
        ),
        ("🖼️ Con imagen sobre el extent", f"{len(present_years)}"),
        (
            "📐 Declaran resolución",
            f"{len(declaring)} años · valores "
            f"{', '.join(f'{v:g}' for v in distinct) or '—'} m",
        ),
        (
            "📨 Resolución pedida al sondear",
            f"{asked} — lo que pedimos, no lo que voló el sensor",
        ),
        (
            "📌 `default_gsd_m` de la entrada",
            f"{GSD_M} m/px — supuesto para la aritmética de teselas, no una medición",
        ),
    ],
    note=(
        "Declarada ≠ medida: medir la resolución nativa exige pedir el mismo bbox a "
        "más píxeles hasta que el detalle deje de crecer, y eso no es de esta sección."
    ),
)
MEASURED["epochs"] = {
    "probed": epoch_table,
    "present_years": present_years,
    "requested_gsd": asked,
    # declarado, no medido: la sección 12 no puede presentarlo como una medición
    "declared_gsd_m_not_measured": {
        year: [d.gsd_m for d in found] for year, found in DECLARED.items() if found
    },
}
if present_years:
    record(
        "5 épocas",
        f"épocas: {len(present_years)} de {len(epoch_table)} años con imagen sobre "
        f"el extent ({present_years[0]}–{present_years[-1]}), {len(declaring)} con "
        f"resolución declarada en su Abstract; pedido a {asked}",
    )
else:
    reasons = sorted({str(row["evidencia"]) for row in epoch_table})
    absent(
        "5 épocas",
        f"ningún año del rango devolvió imagen sobre el extent — {len(epoch_table)} "
        f"sondeos, {len(reasons)} motivo(s) distinto(s), el primero «{reasons[0]}»",
    )

ABSENT: 2004 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m (`Abstract` de la capa, 2 declaración(es))
ABSENT: 2005 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m · 0.4 m (`Abstract` de la capa, 3 declaración(es))
ABSENT: 2007 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m (`Abstract` de la capa, 2 declaración(es))
ABSENT: 2008 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m (`Abstract` de la capa, 2 declaración(es))
ABSENT: 2010 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m · 0.25 m (`Abstract` de la capa, 2 declaración(es))
ABSENT: 2013 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.5 m (`Abstract` de la capa, 1 declaración(es))
ABSENT: 2016 — sin imagen (absent: blank image: no pixel content for this epoch); resolución 0.

,año,anunciada,veredicto,evidencia,blanco,estado,bytes,md5,resolución declarada,fuente de la resolución
0,2004,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
1,2005,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m · 0.4 m,"`Abstract` de la capa, 3 declaración(es)"
2,2006,sí,present,image with pixel content,False,200,195007,9817c876,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
3,2007,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
4,2008,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
5,2009,sí,present,image with pixel content,False,200,200498,eacd3ca7,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
6,2010,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
7,2011,sí,present,image with pixel content,False,200,199029,503520ad,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
8,2012,sí,present,image with pixel content,False,200,52282,a673e0a8,0.5 m · 0.25 m,"`Abstract` de la capa, 2 declaración(es)"
9,2013,sí,absent,blank image: no pixel content for this epoch,True,200,334,cd37744a,0.5 m,"`Abstract` de la capa, 1 declaración(es)"


**Resolución declarada, con las palabras del servicio**

| | |
|---|---|
| 2004 · 0.5 m | «Abarca las Comunidades de Andalucía, Castilla y León, Cataluña, Comunitat Valenciana, Galicia y Murcia con resolución de 0,5m» |
| 2004 · 0.25 m | «y País Vasco y La Rioja con resolución de 0,25 m» |
| 2005 · 0.5 m | «Abarca las Comunidades Autónomas de Andalucía, Cantabria, SW de Castilla y León, Cataluña, Comunitat Valenciana, Extremadura, Galicia y País Vasco con una resolución de 0,5 m» |
| 2005 · 0.25 m | «y la parte NE de Castilla y León con resolución de 0,25 m» |
| 2005 · 0.4 m | «y Canarias con resolución de 0,40 m» |
| 2006 · 0.5 m | «Abarca las Comunidades Autónomas de Andalucía, Aragón, Isalas Baleares, Castilla-La Mancha, NW de CAstilla y León, Cataluña, Extremadura, La Rioja, Madrid, Comunitat Valenciana y las ciudades de Ceuta y Melilla con una resolución de 0,5 m» |
| 2006 · 0.25 m | «y País Vasco, Navarra, SE de Castilla y León y Asturias con una resolución de 0,25 m» |
| 2007 · 0.5 m | «Abarca las Comunidades Autónomas de N de Andalucía, Norte de Castilla y León, Cataluña, SW de Extremadura, País Vasco y Comunitat Valenciana con una resolución de 0,5 m» |
| 2007 · 0.25 m | «y Cantabria, SW de Castilla y León, Se de Extremadura, Galicia y Murcia con resolución de 0,25 m» |
| 2008 · 0.5 m | «Abarca las Comunidades Autónomas de Sur de Andalucía, Baleares, SE de Castilla y León, Cataluña y Comunitat Valenciana a una resolución de 0,50m» |
| 2008 · 0.25 m | «y NW de Castilla y León, Capitales de provincia de Cataluña, Extremadura, Galicia, Navarra y País Vasco a una resolución de 0.25 m» |
| 2009 · 0.5 m | «Abarca las Comunidades Autónomas de Norte de Andalucía, Aragón, SW de Castilla y León, SW y NE de Extremadura y Madrid a una resolución de 0,50m» |
| 2009 · 0.25 m | «y la Comunitat Valenciana, Murcia, La Rioja, Galicia, SE de Extremadura, Cataluña, NE de Castilla y León, Castilla-La Mancha, Islas Canarias y Asturias a una resolución de 0,25 m» |
| 2010 · 0.5 m | «Abarca las Comunidades Autónomas de Sur de Andalucía, NW de Castilla y León, NW de Extremadura a una resolución de 0,50m» |
| 2010 · 0.25 m | «y Las Illes Balears, Cantabria, SE de Castilla y León, Cataluña, NE de Extremadura, Galicia, País Vasco y Comunitat Valenciana a una resolución de 0.25 m» |
| 2011 · 0.5 m | «Abarca las Comunidades Autónomas de Norte de Andalucía, Norte de Castilla y León, SE de Extremadura, y Murcia a una resolución de 0,50 m» |
| 2011 · 0.25 m | «y Sur de Castilla y León, Cataluña, SW de Extremadura, Galicia, Madrid y Asturias a una resolución de 0,25 m» |
| 2012 · 0.5 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, interior de Cataluña, NE de Extremadura, La Rioja y Comunitat Valenciana a una resolución de 0,50 m» |
| 2012 · 0.25 m | «y Las Illes Balears, las Islas Canarias, la costa de Cataluña, Navarra, País Vasco y el NW de Extremadura a una resolución de 0,25 m» |
| 2013 · 0.5 m | «Abarca las Comunidades Autónomas de Andalucía, Extremadura y Murcia a una resolución de 0,50 m» |
| 2014 · 0.25 m | «Abarca las Comunidades Autónomas de Asturias, Cantabria, Galicia, La Rioja, Madrid, País Vasco y Alicante a una resolución de 0,25 m» |
| 2014 · 0.5 m | «, Castilla y León y Navarra a una resolución de 0.50 m» |
| 2014 · 0.1 m | «y las ciudades de Ceuta y Melilla a una resolución de 0.10 m» |
| 2015 · 0.5 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, interior de Cataluña a una resolución de 0,50 m» |
| 2015 · 0.25 m | «y Comunitat Valenciana, Illes Balears, las islas Canarias y la costa de Cataluña a una resolución de 0,25 m» |
| 2016 · 0.5 m | «Abarca las Comunidades Autónomas de Andalucía a una resolución de 0,50 m» |
| 2016 · 0.25 m | «y Extremadura y Murcia a una resolución de 0,25 m» |
| 2017 · 0.25 m | «Abarca las Comunidades Autónomas de Asturias, Cantabria, Castilla y León, Galicia, La Rioja, Madrid, Navarra, País Vasco y Alicante a una resolución de 0,25 m» |
| 2018 · 0.25 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, Cataluña y Comunitat Valenciana a una resolución de 0,25 m» |
| 2018 · 0.15 m | «y las Illes Balears y las islas Canarias a una resolución de 0,15 y 0,16 m» |
| 2018 · 0.16 m | «y las Illes Balears y las islas Canarias a una resolución de 0,15 y 0,16 m» |
| 2019 · 0.25 m | «Abarca las Comunidades Autónomas de Andalucía, Extremadura y Región de Murcia a una resolución de 0,25 m» |
| 2020 · 0.25 m | «Abarca las Comunidades Autónomas de Asturias, Castilla y León, La Rioja, Madrid, País Vasco y Comunidad Foral de Navarra a una resolución de 0,25 m» |
| 2020 · 0.15 m | «y Galicia y Cantabria a 0,15 m» |
| 2021 · 0.25 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, Cataluña, Comunitat Valenciana y las Illes Balears a una resolución de 0,25 m» |
| 2021 · 0.15 m | «y las Islas Canarias a 0,15 m» |
| 2022 · 0.25 m | «Abarca las Comunidades Autónomas de Extremadura, Andalucía y Región de Murcia a una resolución de 0,25 m» |
| 2023 · 0.25 m | «Abarca las Comunidades Autónomas de Castilla y León y Comunidad de Madrid con una resolución de 25 cm» |
| 2023 · 0.15 m | «y La Rioja, Cantabria, País Vasco y Galicia este con una resolución de 15 cm» |
| 2024 · 0.25 m | «Abarca las Comunidades Autónomas de Aragón, Castilla-La Mancha, Cataluña, Comunitat Valenciana y las Illes Balears con una resolución de 25 cm» |
| 2024 · 0.2 m | «y las Islas Canarias con una resolución de 20 cm» |

Cada fila es una declaración, no una medición, y la cita es la región tal como la nombra el servicio. Las capacidades no dicen en qué región cae este extent: lo contesta su capa de cobertura por `GetFeatureInfo`.

**Resumen de la sección 5**

| | |
|---|---|
| 📅 Años del rango | 21 (2004–2024) |
| 🧾 Anunciados por el servicio | 21 |
| 🖼️ Con imagen sobre el extent | 9 |
| 📐 Declaran resolución | 21 años · valores 0.1, 0.15, 0.16, 0.2, 0.25, 0.4, 0.5 m |
| 📨 Resolución pedida al sondear | 5.25 × 3.50 m/px (256×256 px sobre el extent completo) — lo que pedimos, no lo que voló el sensor |
| 📌 `default_gsd_m` de la entrada | 0.25 m/px — supuesto para la aritmética de teselas, no una medición |

Declarada ≠ medida: medir la resolución nativa exige pedir el mismo bbox a más píxeles hasta que el detalle deje de crecer, y eso no es de esta sección.

épocas: 9 de 21 años con imagen sobre el extent (2006–2023), 21 con resolución declarada en su Abstract; pedido a 5.25 × 3.50 m/px (256×256 px sobre el extent completo)


## 6 · La resolución nativa, medida año por año

La sección 5 recoge la resolución **declarada**; esta la **mide**. Sobre una ventana de
`TILE_PX × GSD_M` metros centrada en el extent —una tesela de suelo— se pide el mismo
terreno a cinco tamaños, de 128 a 2048 px, solo en los años presentes.

El estadístico es el **residuo no explicado** entre gradas: se ajusta por mínimos cuadrados
la mejor interpolación lineal de la anterior y se mide lo que sobra, dividido por la
desviación típica de esa grada. Es así una fracción de su propio contraste, sea cual sea el
remuestreo: la grada que solo amplía la anterior se reproduce y cae al suelo de ruido.

El veredicto es la grada más fina sobre el umbral, solo si otra más fina ya está debajo; una
curva que no se aplana se reporta así. El umbral es fijo y sale en cada fila; el selftest lo
barre de 0,02 a 0,20 sin que la cifra cambie.

**Qué mirar:** el `salto` frente al `suelo observado`: sin salto no hay medición.

In [7]:
# --- Sección 6: la resolución nativa, grada a grada sobre una ventana ------
import pandas as pd

WINDOW_SIDE_M = TILE_PX * GSD_M  # una tesela de suelo, con el GSD supuesto de la entrada
LADDER_ROWS: list[dict[str, object]] = []  # una fila por grada pedida, con su md5
RESOLUTION: dict[int, dict[str, object]] = {}  # año -> veredicto de su escalera
RUNGS: dict[int, list[dict[str, object]]] = {}  # año -> sus gradas, para la tabla
STOPS: list[str] = []  # dónde se le acabó la escalera al servicio, si pasó


def no_ladder(year: int) -> None:
    """Dice por qué un año no tiene escalera, con el motivo que dio la sección 5."""
    got = probed.get(year)
    print(
        f"ABSENT: {year} — sin escalera: la sección 5 no lo dio por presente "
        f"({str(got['evidence']) if got else NOT_ASKED})"
    )


def building_px(gsd: float) -> str:
    """Cuántos píxeles ocupa el edificio SUPUESTO de BUILDING_SIDE_M a esa resolución."""
    side = BUILDING_SIDE_M / gsd
    return f"{side:.0f}×{side:.0f} px ({side * side:.0f} px²)"


if not present_years:
    for year in YEARS:
        no_ladder(year)
    absent(
        "6 resolución",
        f"ninguna época con imagen sobre el extent: no hay ventana que medir "
        f"({len(YEARS)} años sondeados en la sección 5)",
    )
else:
    WINDOW = eda.centred_window(EXTENT_BBOX, WINDOW_SIDE_M)
    side_x, side_y = WINDOW[2] - WINDOW[0], WINDOW[3] - WINDOW[1]
    seen = [int(row["nbytes"]) for row in EPOCH_ROWS if row["verdict"] == "present"]
    per_px = sum(seen) / len(seen) / (eda.PROBE_SIZE[0] * eda.PROBE_SIZE[1])
    planned = len(present_years) * len(eda.LADDER_PX)
    cost = per_px * sum(px * px for px in eda.LADDER_PX) * len(present_years)
    panel(
        "El plan, antes de pedir nada",
        [
            (
                "🪟 Ventana",
                f"{side_x:.1f} × {side_y:.1f} m centrados en el extent"
                f" (= {TILE_PX} px × {GSD_M} m/px supuestos)",
            ),
            (
                "🪜 Gradas",
                " · ".join(f"{px}px→{side_x / px:.3f}" for px in eda.LADDER_PX) + " m/px",
            ),
            (
                "📨 Peticiones",
                f"{len(present_years)} años × {len(eda.LADDER_PX)} gradas ="
                f" {planned} como máximo",
            ),
            (
                "⚖️ Coste estimado",
                f"~{cost / (1024 * 1024):.0f} MB, extrapolados de los"
                f" {per_px:.2f} bytes/píxel que midió la sección 5",
            ),
            (
                "🏠 Edificio",
                f"{BUILDING_SIDE_M:g} m de lado — supuesto declarado en la sección 2;"
                " el reparto real de huellas llega en la sección 8",
            ),
            (
                "📏 Umbral de detalle",
                f"{eda.DETAIL_FLOOR:g} del contraste de la ventana, constante",
            ),
        ],
        note=(
            "La escalera se detiene en la primera grada que el servicio no sirva: ese "
            "techo es un hallazgo, no un fallo. Nada de esto se ha pedido todavía."
        ),
    )
    for year in YEARS:
        if year not in present_years:
            no_ladder(year)
            continue
        rungs = eda.resolution_ladder(
            PROVIDER,
            PROVIDER.epoch_layer_pattern.format(year=year),
            WINDOW,
            EXTENT_IN,
            OUT_DIR,
        )
        found = eda.resolution_from_ladder(rungs)
        RESOLUTION[year] = found
        RUNGS[year] = rungs
        for rung in rungs:
            stopped = bool(rung["stopped"])
            LADDER_ROWS.append(
                {
                    "año": year,
                    "px": rung["size_px"],
                    "m/px pedidos": round(float(str(rung["gsd_m"])), 4),
                    "estado": rung["status"],
                    "bytes": rung["nbytes"],
                    "md5": str(rung["md5"])[:8],
                    "residuo": rung["share"],
                    "muestras": rung["samples"],
                    "parada": "sí" if stopped else "—",
                    "motivo": rung["reason"],
                }
            )
            if stopped:
                STOPS.append(f"{year} en {rung['size_px']} px — {rung['reason']}")
        if found["measured_gsd_m"] is None:
            print(f"ABSENT: {year} — sin cifra medida: {found['reason']}")

    verdicts: list[dict[str, object]] = []
    for year, found in RESOLUTION.items():
        measured = found["measured_gsd_m"]
        curve = found["curve"] if isinstance(found["curve"], list) else []
        bound = ""
        if found["bound"] == "no_finer_than":
            bound = f"≥ {float(str(found['bound_gsd_m'])):g} m/px"
        elif found["bound"] == "at_least_as_fine_as":
            bound = f"≤ {float(str(found['bound_gsd_m'])):g} m/px"
        verdicts.append(
            {
                "año": year,
                "residuo por grada": " → ".join(f"{s:.3f}" for _, _, s in curve) or "—",
                "umbral": eda.DETAIL_FLOOR,
                "medida m/px": f"{float(str(measured)):g}"
                if measured is not None
                else "—",
                "salto": f"×{float(str(found['drop'])):.1f}"
                if found["drop"] is not None
                else "—",
                "suelo observado": f"{float(str(found['floor_observed'])):.3f}"
                if found["floor_observed"] is not None
                else "—",
                "px por edificio": building_px(float(str(measured)))
                if measured is not None
                else "—",
                "cota": bound,
                "motivo": found["reason"],
            }
        )
    display(pd.DataFrame(verdicts))
    display(pd.DataFrame(LADDER_ROWS))

    measured_years = {
        year: float(str(found["measured_gsd_m"]))
        for year, found in RESOLUTION.items()
        if found["measured_gsd_m"] is not None
    }
    distinct = sorted(set(measured_years.values()))
    served = [int(str(row["px"])) for row in LADDER_ROWS if row["parada"] == "—"]
    for stop in STOPS:
        print(f"HALLAZGO: la escalera se detuvo en {stop}")
    if len(distinct) > 1:
        print(
            "HALLAZGO: la resolución medida no es la misma en todos los años — "
            + ", ".join(
                f"{value:g} m/px en {sum(1 for v in measured_years.values() if v == value)}"
                f" año(s)"
                for value in distinct
            )
            + "; el servicio declara varias resoluciones según año y región, así que una "
            "diferencia por año es un hallazgo, no algo que promediar"
        )
    if measured_years:
        same = sum(1 for v in measured_years.values() if abs(v - GSD_M) < 1e-9)
        print(
            f"NOTA: {same} de {len(measured_years)} años medidos coinciden con el "
            f"supuesto GSD_M = {GSD_M} m/px de la entrada del proveedor. El supuesto "
            "sigue gobernando la aritmética de teselas y el extent: esta sección mide, "
            "no promueve."
        )
    panel(
        "Resumen de la sección 6",
        [
            (
                "🪟 Ventana medida",
                f"{side_x:.1f} × {side_y:.1f} m"
                + (
                    " (cuadrada)"
                    if abs(side_x - side_y) < 1e-6
                    else " — RECORTADA al extent: cada eje lleva su propia resolución"
                ),
            ),
            ("📨 Peticiones hechas", f"{len(LADDER_ROWS)} de {planned} planificadas"),
            (
                "🪜 Grada más grande servida",
                f"{max(served)} px" if served else "ninguna",
            ),
            (
                "📐 Años con cifra medida",
                f"{len(measured_years)} de {len(present_years)} presentes",
            ),
            (
                "📊 Valores medidos",
                ", ".join(f"{v:g}" for v in distinct) + " m/px" if distinct else "—",
            ),
            (
                "🏠 Píxeles por edificio",
                f"{building_px(max(distinct))} con el peor valor medido"
                if distinct
                else "—",
            ),
            (
                "📌 Supuesto en uso",
                f"GSD_M = {GSD_M} m/px, de la entrada del proveedor, sin promover",
            ),
        ],
        note=(
            "Cada grada es una fila del manifiesto: su md5 y sus bytes están en la "
            "segunda tabla, y sin esa fila la cifra no sería una medición."
        ),
    )
    MEASURED["resolution"] = {
        "window_m": [side_x, side_y],
        "ladder_px": list(eda.LADDER_PX),
        "detail_floor": eda.DETAIL_FLOOR,
        "building_side_m_assumed": BUILDING_SIDE_M,
        "measured_gsd_m": measured_years,
        "rungs": LADDER_ROWS,
    }
    if measured_years:
        record(
            "6 resolución",
            f"resolución medida en {len(measured_years)} de {len(present_years)} años "
            f"presentes: {', '.join(f'{v:g}' for v in distinct)} m/px sobre una ventana "
            f"de {side_x:.0f} m, {len(LADDER_ROWS)} peticiones; "
            f"{building_px(max(distinct))} por edificio supuesto de {BUILDING_SIDE_M:g} m",
        )
    else:
        reasons = sorted({str(found["reason"]) for found in RESOLUTION.values()})
        absent(
            "6 resolución",
            f"ninguna escalera dio una cifra sobre {len(present_years)} años presentes "
            f"— {len(reasons)} motivo(s) distinto(s), el primero «{reasons[0]}»",
        )

# La tabla a disco: es lo que deja a la sección 6.2 dibujar la escalera en un
# kernel frío, sin repetir la medición. Con cabecera incluso si está vacía.
LADDER_TABLE = [
    row
    for year, rungs in RUNGS.items()
    for row in eda.ladder_rows(year, rungs, RESOLUTION[year])
]
LADDER_PATH = eda.write_ladder(LADDER_TABLE, OUT_DIR)
print(f"tabla de gradas: {len(LADDER_TABLE)} filas en {LADDER_PATH.relative_to(HERE)}")


**El plan, antes de pedir nada**

| | |
|---|---|
| 🪟 Ventana | 64.0 × 64.0 m centrados en el extent (= 256 px × 0.25 m/px supuestos) |
| 🪜 Gradas | 128px→0.500 · 256px→0.250 · 512px→0.125 · 1024px→0.062 · 2048px→0.031 m/px |
| 📨 Peticiones | 9 años × 5 gradas = 45 como máximo |
| ⚖️ Coste estimado | ~121 MB, extrapolados de los 2.52 bytes/píxel que midió la sección 5 |
| 🏠 Edificio | 10 m de lado — supuesto declarado en la sección 2; el reparto real de huellas llega en la sección 8 |
| 📏 Umbral de detalle | 0.04 del contraste de la ventana, constante |

La escalera se detiene en la primera grada que el servicio no sirva: ese techo es un hallazgo, no un fallo. Nada de esto se ha pedido todavía.

ABSENT: 2004 — sin escalera: la sección 5 no lo dio por presente (blank image: no pixel content for this epoch)
ABSENT: 2005 — sin escalera: la sección 5 no lo dio por presente (blank image: no pixel content for this epoch)
ABSENT: 2006 — sin cifra medida: no rung added detail above the floor: the native sampling is no finer than the coarsest rung asked for
ABSENT: 2007 — sin escalera: la sección 5 no lo dio por presente (blank image: no pixel content for this epoch)
ABSENT: 2008 — sin escalera: la sección 5 no lo dio por presente (blank image: no pixel content for this epoch)
ABSENT: 2010 — sin escalera: la sección 5 no lo dio por presente (blank image: no pixel content for this epoch)
ABSENT: 2012 — sin cifra medida: fewer than two readable rungs: nothing to compare
ABSENT: 2013 — sin escalera: la sección 5 no lo dio por presente (blank image: no pixel content for this epoch)
ABSENT: 2015 — sin cifra medida: fewer than two readable rungs: nothing to compare
ABSENT: 2016 — sin escaler

,año,residuo por grada,umbral,medida m/px,salto,suelo observado,px por edificio,cota,motivo
0,2006,0.000 → 0.000 → 0.000 → 0.000,0.04,—,—,0.000,—,≥ 0.5 m/px,no rung added detail above the floor: the nati...
1,2009,0.059 → 0.000 → 0.000 → 0.000,0.04,0.25,×33522205206079.4,0.000,40×40 px (1600 px²),,
2,2011,0.109 → 0.000 → 0.000 → 0.000,0.04,0.25,×60603968999589.7,0.000,40×40 px (1600 px²),,
3,2012,—,0.04,—,—,—,—,,fewer than two readable rungs: nothing to compare
4,2014,0.085 → 0.000 → 0.000 → 0.000,0.04,0.25,×118173387839954.8,0.000,40×40 px (1600 px²),,
5,2015,—,0.04,—,—,—,—,,fewer than two readable rungs: nothing to compare
6,2017,0.099 → 0.000 → 0.000 → 0.000,0.04,0.25,×73319049599461.7,0.000,40×40 px (1600 px²),,
7,2020,0.184 → 0.000 → 0.000 → 0.000,0.04,0.25,×92078991470716.7,0.000,40×40 px (1600 px²),,
8,2023,0.186 → 0.000 → 0.000 → 0.000,0.04,0.25,×208727232341073.6,0.000,40×40 px (1600 px²),,


,año,px,m/px pedidos,estado,bytes,md5,residuo,muestras,parada,motivo
0,2006,128,0.5000,200,46543,39c27d28,NaN,0,—,
1,2006,256,0.2500,200,54954,f2892ad1,8.889706e-16,61504,—,
2,2006,512,0.1250,200,62521,9dbfc5e2,1.496531e-15,200000,—,
3,2006,1024,0.0625,200,87220,66d0e787,2.427308e-15,200000,—,
4,2006,2048,0.0312,200,174703,71f6cd62,1.570899e-15,200000,—,
5,2009,128,0.5000,200,50331,06364428,NaN,0,—,
6,2009,256,0.2500,200,185121,a8cb23c4,5.863473e-02,61504,—,
7,2009,512,0.1250,200,215839,7d710231,1.749131e-15,200000,—,
8,2009,1024,0.0625,200,246448,e524fd45,1.027605e-15,200000,—,
9,2009,2048,0.0312,200,348560,f4f79b01,1.250934e-15,200000,—,


HALLAZGO: la escalera se detuvo en 2012 en 128 px — absent: blank image: no pixel content for this epoch
HALLAZGO: la escalera se detuvo en 2015 en 128 px — absent: blank image: no pixel content for this epoch
NOTA: 6 de 6 años medidos coinciden con el supuesto GSD_M = 0.25 m/px de la entrada del proveedor. El supuesto sigue gobernando la aritmética de teselas y el extent: esta sección mide, no promueve.


**Resumen de la sección 6**

| | |
|---|---|
| 🪟 Ventana medida | 64.0 × 64.0 m (cuadrada) |
| 📨 Peticiones hechas | 37 de 45 planificadas |
| 🪜 Grada más grande servida | 2048 px |
| 📐 Años con cifra medida | 6 de 9 presentes |
| 📊 Valores medidos | 0.25 m/px |
| 🏠 Píxeles por edificio | 40×40 px (1600 px²) con el peor valor medido |
| 📌 Supuesto en uso | GSD_M = 0.25 m/px, de la entrada del proveedor, sin promover |

Cada grada es una fila del manifiesto: su md5 y sus bytes están en la segunda tabla, y sin esa fila la cifra no sería una medición.

resolución medida en 6 de 9 años presentes: 0.25 m/px sobre una ventana de 64 m, 37 peticiones; 40×40 px (1600 px²) por edificio supuesto de 10 m
tabla de gradas: 37 filas en out/ladder.csv


### 6.1 · La calidad de cada época, como imagen

Arriba, a qué resolución llega cada año; aquí, si la imagen se puede mirar. De cada época de
la sección 5 salen tamaño, bandas, media y desviación típica, `saturated_share`,
`nodata_share` y `gradient_variance` —un indicador de nitidez—, y tres marcas: **blanca**,
**saturada** y **bajo contraste**. Cada una lleva pegado el umbral que la produjo: un umbral
invisible no se puede discutir. Debajo, la rejilla de miniaturas de la más antigua a la más
reciente, cada una con su año y su md5; los años sin imagen salen antes, uno por línea, con
su motivo.

**El límite de esta sección, dicho aquí para que no se tome por otra cosa:** juzga la imagen
*como imagen*, al tamaño en que se pidió, y no dice nada sobre si un edificio es legible en
ella. Eso lo deciden los píxeles por edificio de la sección 6 y los chips de la sección 10.

**Qué mirar:** la columna `defectos` y la rejilla juntas: un año marcado, o una miniatura
vacía, queda fuera de cualquier comparación.

In [8]:
# --- Sección 6.1: la calidad de cada época, juzgada como imagen ------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SATURATED_MAX = 0.05  # umbral: muestras en el máximo del tipo por encima de esto
LOW_CONTRAST_STD = 10.0  # umbral: desviación típica media por banda, en niveles digitales
THUMBS_PER_ROW = 6  # cuántas miniaturas por fila de la rejilla
QUALITY_ROWS: list[dict[str, object]] = []  # una fila por época con imagen
THUMBS: list[tuple[int, object, str]] = []  # (año, miniatura, md5) para la rejilla

with_image = {
    int(row["year"]): row
    for row in EPOCH_ROWS
    if row["path"] and row["blank"] is not None
}
for year in YEARS:  # los años sin imagen salen primero, uno por línea y con su motivo
    if year not in with_image:
        got = probed.get(year)
        print(
            f"ABSENT: {year} — sin imagen que juzgar "
            f"({str(got['evidence']) if got else NOT_ASKED})"
        )

if not with_image:
    absent(
        "6.1 calidad",
        f"ninguna de las {len(YEARS)} épocas del rango dejó una imagen legible: no hay "
        "nada que medir como imagen",
    )
else:
    for year in sorted(with_image):
        row = with_image[year]
        path = Path(str(row["path"]))
        try:
            stats = eda.image_stats(path)
            pixels, alpha = eda._read_image(path)
        except Exception as exc:  # el motivo se imprime, no se esconde
            print(f"ABSENT: {year} — imagen ilegible: {type(exc).__name__}: {exc}")
            continue
        colour = [band for band in range(int(str(stats["bands"]))) if band != alpha]
        spread = sum(float(stats["std"][band]) for band in colour) / len(colour)
        saturated = float(str(stats["saturated_share"]))
        marks = []
        if row["blank"]:
            marks.append("blanca (alfa vacío o un único valor en toda la imagen)")
        if saturated >= SATURATED_MAX:
            marks.append(f"saturada ({saturated:.3f} ≥ {SATURATED_MAX})")
        if spread < LOW_CONTRAST_STD:
            marks.append(f"bajo contraste (σ {spread:.1f} < {LOW_CONTRAST_STD:g})")
        QUALITY_ROWS.append(
            {
                "año": year,
                "tamaño": f"{stats['width']}×{stats['height']}",
                "bandas": stats["bands"],
                "media": round(
                    sum(float(stats["mean"][band]) for band in colour) / len(colour), 1
                ),
                "σ": round(spread, 1),
                "saturated_share": round(saturated, 4),
                "nodata_share": round(float(str(stats["nodata_share"])), 4),
                "gradient_variance": round(float(str(stats["gradient_variance"])), 2),
                "md5": str(row["md5"])[:8],
                "defectos": " · ".join(marks) or "ok",
            }
        )
        band_stack = np.asarray(pixels[colour[:3]])
        THUMBS.append(
            (
                year,
                np.moveaxis(band_stack, 0, -1) if len(colour) >= 3 else band_stack[0],
                str(row["md5"])[:8],
            )
        )
    display(pd.DataFrame(QUALITY_ROWS))

    if THUMBS:
        columns = min(THUMBS_PER_ROW, len(THUMBS))
        rows_of_grid = (len(THUMBS) + columns - 1) // columns
        figure, axes = plt.subplots(
            rows_of_grid,
            columns,
            figsize=(2.1 * columns, 2.4 * rows_of_grid),
            squeeze=False,
        )
        for axis, (year, thumb, digest) in zip(axes.ravel(), THUMBS, strict=False):
            axis.imshow(thumb, cmap=None if np.ndim(thumb) == 3 else "gray")
            axis.set_title(f"{year} · {digest}", fontsize=8)
        for axis in axes.ravel():
            axis.set_axis_off()
        figure.suptitle(
            f"Épocas con imagen, de la más antigua a la más reciente — "
            f"{len(THUMBS)} de {len(YEARS)} años, a {eda.PROBE_SIZE[0]} px sobre el "
            f"extent completo",
            fontsize=10,
        )
        figure.tight_layout()
        # A disco y mostrada desde ahí: dentro del .ipynb esta rejilla pesaba un
        # megabyte en base64 y se reescribía entera en git en cada ejecución.
        GRID_PATH = OUT_DIR / "figs" / "epochs.png"
        figure.savefig(GRID_PATH, dpi=110, bbox_inches="tight")
        plt.close(figure)
        # Referencia markdown, no `Image(filename=…)`: eso incrusta el PNG en el
        # output y devuelve el megabyte al .ipynb. Lab la resuelve desde disco.
        display(Markdown(f"![Rejilla de épocas]({GRID_PATH.relative_to(HERE)})"))
        print(f"rejilla guardada en {GRID_PATH.relative_to(HERE)}")

    marked = [row for row in QUALITY_ROWS if row["defectos"] != "ok"]
    counted = {
        name: sum(1 for row in QUALITY_ROWS if name in str(row["defectos"]))
        for name in ("blanca", "saturada", "bajo contraste")
    }
    panel(
        "Resumen de la sección 6.1",
        [
            (
                "🖼️ Épocas con imagen",
                f"{len(QUALITY_ROWS)} de {len(YEARS)} años del rango",
            ),
            (
                "⬜ Blancas",
                f"{counted['blanca']} · umbral: alfa vacío o un único valor",
            ),
            (
                "🔆 Saturadas",
                f"{counted['saturada']} · umbral: saturated_share ≥ {SATURATED_MAX}",
            ),
            (
                "🌫️ Bajo contraste",
                f"{counted['bajo contraste']} · umbral: σ media < {LOW_CONTRAST_STD:g}",
            ),
            (
                "📨 Tamaño juzgado",
                f"{eda.PROBE_SIZE[0]}×{eda.PROBE_SIZE[1]} px sobre el extent completo, "
                "tal como lo pidió la sección 5",
            ),
        ],
        note=(
            "Estas cifras juzgan la imagen **como imagen**, al tamaño en que se pidió. "
            "No dicen si un edificio es legible en ella: eso son los píxeles por "
            "edificio de la sección 6 y los chips de la sección 10."
        ),
    )
    MEASURED["quality"] = {
        "thresholds": {
            "saturated_share_max": SATURATED_MAX,
            "low_contrast_std_min": LOW_CONTRAST_STD,
        },
        "probe_px": list(eda.PROBE_SIZE),
        "epochs": QUALITY_ROWS,
    }
    record(
        "6.1 calidad",
        f"calidad: {len(QUALITY_ROWS)} épocas con imagen de {len(YEARS)} años, "
        f"{len(marked)} marcadas ({counted['blanca']} blancas, "
        f"{counted['saturada']} saturadas, {counted['bajo contraste']} con bajo "
        f"contraste) con umbrales saturated_share ≥ {SATURATED_MAX} y σ < "
        f"{LOW_CONTRAST_STD:g}",
    )

,año,tamaño,bandas,media,σ,saturated_share,nodata_share,gradient_variance,md5,defectos
0,2004,256×256,4,0.0,0.0,0.0000,1.0000,0.00,cd37744a,blanca (alfa vacío o un único valor en toda la...
1,2005,256×256,4,0.0,0.0,0.0000,1.0000,0.00,cd37744a,blanca (alfa vacío o un único valor en toda la...
2,2006,256×256,4,113.4,36.7,0.0000,0.0000,216.52,9817c876,ok
3,2007,256×256,4,0.0,0.0,0.0000,1.0000,0.00,cd37744a,blanca (alfa vacío o un único valor en toda la...
4,2008,256×256,4,0.0,0.0,0.0000,1.0000,0.00,cd37744a,blanca (alfa vacío o un único valor en toda la...
5,2009,256×256,4,90.3,38.6,0.0000,0.0000,280.33,eacd3ca7,ok
6,2010,256×256,4,0.0,0.0,0.0000,1.0000,0.00,cd37744a,blanca (alfa vacío o un único valor en toda la...
7,2011,256×256,4,141.4,38.2,0.0000,0.0000,237.88,503520ad,ok
8,2012,256×256,4,27.4,51.8,0.0000,0.7529,179.02,a673e0a8,ok
9,2013,256×256,4,0.0,0.0,0.0000,1.0000,0.00,cd37744a,blanca (alfa vacío o un único valor en toda la...


![Rejilla de épocas](out/figs/epochs.png)

rejilla guardada en out/figs/epochs.png


**Resumen de la sección 6.1**

| | |
|---|---|
| 🖼️ Épocas con imagen | 21 de 21 años del rango |
| ⬜ Blancas | 12 · umbral: alfa vacío o un único valor |
| 🔆 Saturadas | 0 · umbral: saturated_share ≥ 0.05 |
| 🌫️ Bajo contraste | 12 · umbral: σ media < 10 |
| 📨 Tamaño juzgado | 256×256 px sobre el extent completo, tal como lo pidió la sección 5 |

Estas cifras juzgan la imagen **como imagen**, al tamaño en que se pidió. No dicen si un edificio es legible en ella: eso son los píxeles por edificio de la sección 6 y los chips de la sección 10.

calidad: 21 épocas con imagen de 21 años, 12 marcadas (12 blancas, 0 saturadas, 12 con bajo contraste) con umbrales saturated_share ≥ 0.05 y σ < 10


## 6.2 · La escalera, vista

La sección 6 dio una cifra por año; esta la enseña. Lee `out/ladder.csv` y los cuerpos
que están en `out/epochs/`, emparejados por su md5, así que **se ejecuta sola en un kernel
recién arrancado**: no hay que repetir la medición de 45 peticiones para volver a mirar
las imágenes.

Una fila por año, una columna por grada, de la más gruesa a la más fina, y todas las
miniaturas dibujadas **al mismo tamaño en pantalla**. Ahí está el punto: pasado el píxel
nativo las imágenes dejan de ganar nitidez mientras el número de píxeles y los bytes
siguen subiendo. Cada miniatura lleva sus píxeles pedidos, sus m/px, sus bytes y su md5,
que es la fila del manifiesto que la respalda.

La figura se guarda en `out/figs/` y se muestra desde disco, para no meter otro megabyte
de base64 dentro del notebook.

**Qué mirar:** a partir de qué columna las imágenes de una misma fila ya no se ven más
nítidas — esa columna es la resolución real de ese año.


In [9]:
# --- Sección 6.2: las imágenes de cada grada, de los últimos años medidos ---
import matplotlib.pyplot as plt

LADDER = eda.read_ladder(OUT_DIR)  # None: la sección 6 no se ha ejecutado
if LADDER is None:
    absent("6.2 escalera vista", "no hay `ladder.csv`: ejecuta antes la sección 6")
elif not LADDER:
    absent("6.2 escalera vista", "`ladder.csv` no tiene filas: la sección 6 no midió ninguna grada")
else:
    by_year: dict[int, list[dict[str, str]]] = {}
    for row in LADDER:
        by_year.setdefault(int(row["year"]), []).append(row)
    for rows in by_year.values():
        rows.sort(key=lambda row: int(row["size_px"]))
    shown = sorted(by_year)[-LADDER_YEARS_SHOWN:]
    columns = max(len(by_year[year]) for year in shown)
    figure, axes = plt.subplots(
        len(shown),
        columns,
        figsize=(2.7 * columns, 3.1 * len(shown)),
        squeeze=False,
    )
    for axis in axes.ravel():
        axis.set_axis_off()
    gone: list[str] = []
    for line, year in enumerate(shown):
        for column, row in enumerate(by_year[year]):
            axis = axes[line][column]
            body = eda.body_for(OUT_DIR, row["md5"])  # el md5 es el nombre del fichero
            if body is None:  # la fila existe, el cuerpo no: se dice, no se dibuja
                gone.append(f"{year} · {row['size_px']}px · md5 {row['md5'][:8]}")
                axis.text(0.5, 0.5, "cuerpo\nausente", ha="center", va="center", fontsize=8)
            else:
                axis.imshow(plt.imread(body))
            axis.set_title(
                f"{year} · {row['size_px']} px · {float(row['gsd_m']):g} m/px\n"
                f"{int(row['nbytes']):,} B · {row['md5'][:8]}",
                fontsize=7,
            )
    figure.suptitle(
        "Misma ventana de suelo, más píxeles por grada: donde deja de ganar nitidez "
        "está el píxel nativo",
        fontsize=10,
    )
    figure.tight_layout()
    LADDER_FIG = OUT_DIR / "figs" / "ladder.png"
    figure.savefig(LADDER_FIG, dpi=110, bbox_inches="tight")
    plt.close(figure)
    # Referencia, no bytes: `Image(filename=…)` incrustaría el PNG en el output.
    display(Markdown(f"![Escalera de resolución]({LADDER_FIG.relative_to(HERE)})"))
    for line in gone:
        print(f"ABSENT: {line} — la fila está en la tabla y el cuerpo no está en disco")
    verdict_rows = [
        (
            f"{year}",
            f"{float(by_year[year][0]['measured_gsd_m']):g} m/px medidos"
            if by_year[year][0]["measured_gsd_m"]
            else f"sin cifra — {by_year[year][0]['measured_reason']}",
        )
        for year in shown
    ]
    panel(
        f"Los {len(shown)} últimos años con escalera",
        verdict_rows,
        note=(
            f"Figura en `{LADDER_FIG.relative_to(HERE)}`, fuera de git porque se vuelve "
            "a dibujar. Los bytes y el md5 de cada grada son su fila del manifiesto."
        ),
    )
    record(
        "6.2 escalera vista",
        f"escalera dibujada para {len(shown)} años ({', '.join(str(y) for y in shown)}), "
        f"{sum(len(by_year[y]) for y in shown)} gradas"
        + (f", {len(gone)} cuerpo(s) ausente(s)" if gone else ""),
    )


![Escalera de resolución](out/figs/ladder.png)

**Los 3 últimos años con escalera**

| | |
|---|---|
| 2017 | 0.25 m/px medidos |
| 2020 | 0.25 m/px medidos |
| 2023 | 0.25 m/px medidos |

Figura en `out/figs/ladder.png`, fuera de git porque se vuelve a dibujar. Los bytes y el md5 de cada grada son su fila del manifiesto.

escalera dibujada para 3 años (2017, 2020, 2023), 15 gradas


## 7 · ¿Llega infrarrojo? Y si no, qué queda en su lugar

El infrarrojo cercano es la banda con la que se calcula el NDVI. Esta sección pregunta dos
cosas: al catálogo, si alguna capa menciona infrarrojo, falso color o un índice; y a los
bytes que ya están en disco, cuántas bandas de color trajo una imagen servida. Lo segundo
zanja lo que el catálogo sólo sugiere.

Después la consecuencia, acotada: **por esta vía** no hay NDVI, y la vegetación por
edificio sale de sustitutos RGB, que confunden plantas con cubiertas verdes y suelo en
sombra. Acotada porque el infrarrojo a esa escala existe en España, pero no por los
servicios que este notebook interroga.

De ahí la última tabla: productos declarados por sus catálogos, **nunca medidos aquí**, con
sus píxeles por edificio al lado y sus incógnitas marcadas `PENDIENTE`.

**Qué mirar:** las bandas de color medidas. Si son tres, no hay infrarrojo y el NDVI por
edificio queda descartado aquí, diga lo que diga un catálogo.


In [10]:
# --- Sección 7: el alcance espectral, contra el catálogo y contra los bytes ---
import pandas as pd

CATALOGUES = globals().get("CAPS", {})  # los catálogos que leyó la sección 4
SPECTRAL: list[dict[str, object]] = []  # capas sospechosas de traer infrarrojo
if not CATALOGUES:
    absent("7 catálogo", "sin catálogos en memoria: ejecuta antes la sección 4")
    SEARCHED = 0
else:
    SEARCHED = sum(len(caps.layers) for caps in CATALOGUES.values())
    for service, caps in CATALOGUES.items():
        for layer, token in eda.spectral_candidates(caps):
            SPECTRAL.append({"servicio": service, "capa": layer, "término": token})
    if SPECTRAL:
        display(pd.DataFrame(SPECTRAL))
        record(
            "7 catálogo",
            f"{len(SPECTRAL)} capa(s) de {SEARCHED} mencionan infrarrojo o un índice; "
            "candidatas, no confirmadas: lo confirman las bandas servidas",
        )
    else:
        absent(
            "7 catálogo",
            f"ninguna de las {SEARCHED} capas anunciadas menciona infrarrojo, falso color "
            f"ni índice espectral — {len(eda.SPECTRAL_TOKENS)} términos buscados en nombre, "
            "título y descripción",
        )

# Lo que de verdad llegó: se cuentan las bandas de una imagen que ya está en disco,
# así que esto no pide nada a la red y no depende de ninguna afirmación.
BANDS = 0
ALPHA = False
MEASURED_GSD = 0.0
LADDER = eda.read_ladder(OUT_DIR)
if not LADDER:
    absent("7 bandas", "sin `ladder.csv`: no hay imagen servida de la que contar bandas")
else:
    finest = max(LADDER, key=lambda row: (int(row["year"]), int(row["size_px"])))
    body = eda.body_for(OUT_DIR, finest["md5"])
    figures = [float(row["measured_gsd_m"]) for row in LADDER if row["measured_gsd_m"]]
    MEASURED_GSD = max(figures) if figures else 0.0
    if body is None:
        absent("7 bandas", f"el cuerpo md5 {finest['md5'][:8]} no está en disco")
    else:
        BANDS, ALPHA = eda.colour_bands(body)
        record(
            "7 bandas",
            f"{BANDS} bandas de color {'+ alfa ' if ALPHA else ''}en la imagen de "
            f"{finest['year']} a {finest['size_px']} px (md5 {finest['md5'][:8]})",
        )

def miles(value: float) -> str:
    """Separador de miles a la española: 1.600, no 1,600."""
    return f"{value:,.0f}".replace(",", ".")

# La consecuencia, escrita y no insinuada: es el entregable de esta sección.
GSD_USED = MEASURED_GSD or GSD_M
GSD_LABEL = "medidos" if MEASURED_GSD else "supuestos"
ortho_px = (BUILDING_SIDE_M / GSD_USED) ** 2 if GSD_USED else 0.0
panel(
    "Consecuencia para los indicadores de vegetación",
    [
        (
            "🛰️ Bandas que llegan",
            f"{BANDS} de color{' + alfa' if ALPHA else ''} — sin infrarrojo"
            if BANDS and BANDS <= 3
            else (
                f"{BANDS} de color — más de tres: revisar si una es infrarrojo"
                if BANDS
                else "sin medir: no había imagen en disco de la que contarlas"
            ),
        ),
        (
            "🌿 NDVI por esta vía",
            "**descartado**: la ortofoto servida no trae infrarrojo"
            if BANDS and BANDS <= 3
            else "sin determinar: hacen falta las bandas servidas",
        ),
        (
            "🔁 Lo que queda por edificio, aquí",
            "sustitutos RGB (exceso de verde, VARI): confunden plantas con cubiertas "
            "verdes y con suelo en sombra, así que un positivo no es vegetación hasta "
            "verificarlo a ojo",
        ),
        (
            "🏠 Píxeles por edificio en la ortofoto",
            f"{miles(ortho_px)} px² a {GSD_USED:g} m/px ({GSD_LABEL}), "
            f"para {BUILDING_SIDE_M:g} m de lado",
        ),
    ],
    note=(
        "Sin infrarrojo, la vegetación sobre una cubierta sólo se puede insinuar con "
        "color visible. Eso no invalida la detección de abandono: la desplaza hacia el "
        "cambio estructural entre épocas, que es lo que sí mide esta resolución."
    ),
)

# Lo que existe fuera de estos servicios: declaraciones de catálogo, sin una sola
# petición detrás. La tabla pone al lado la cifra que decide si sirven.
PRODUCTS = eda.DECLARED_PRODUCTS.get(PROVIDER_KEY, ())
if not PRODUCTS:
    absent("7 productos declarados", "la entrada del proveedor no declara ningún producto")
else:
    display(
        pd.DataFrame(
            [
                {
                    "producto": product.label,
                    "bandas": product.bands,
                    "m/px declarados": f"{product.gsd_m[0]:g}"
                    if product.gsd_m[0] == product.gsd_m[1]
                    else f"{product.gsd_m[0]:g}–{product.gsd_m[1]:g}",
                    "px² por edificio (peor caso)": round(
                        (BUILDING_SIDE_M / product.gsd_m[1]) ** 2, 2
                    ),
                    "años": product.years,
                    "cadencia": product.cadence,
                    "formato": product.file_format,
                    "acceso": product.access,
                    "incógnitas": len(product.unknowns),
                }
                for product in PRODUCTS
            ]
        )
    )
    # Cada incógnita, visible: es lo que tendría que resolverse para que cuente.
    for product in PRODUCTS:
        for unknown in product.unknowns:
            print(f"PENDIENTE: {product.label} — {unknown}")
    fine = [product for product in PRODUCTS if product.gsd_m[1] <= GSD_USED * 2]
    panel(
        "Lo que decide si un producto sirve: píxeles por edificio",
        [
            (
                "🏠 En la ortofoto ya medida",
                f"{miles(ortho_px)} px² a {GSD_USED:g} m/px ({GSD_LABEL})",
            ),
            (
                "🔍 Para juzgar un tejado",
                "hacen falta cientos de px², y eso sólo lo dan los productos de "
                f"decímetros: {len(fine)} de {len(PRODUCTS)} declarados",
            ),
            (
                "🌾 Para el entorno de la parcela",
                "bastan decenas, así que ahí sí entran los de 10 m, y además aportan "
                "varias observaciones al año frente a una por época",
            ),
            (
                "🚪 Cómo se accede",
                "ninguno se sirve por OGC: todos son descarga, la vía que este "
                "instrumento no tiene",
            ),
            (
                "⏳ Qué falta para que cuenten",
                f"{sum(len(product.unknowns) for product in PRODUCTS)} incógnitas, "
                "listadas arriba como PENDIENTE",
            ),
        ],
        note=(
            "Declarado por sus catálogos y **no verificado aquí**: ninguna fila de "
            "`manifest.csv` lo respalda. El infrarrojo a escala de edificio existe en "
            "España; lo que no existe es por los servicios que este notebook interroga."
        ),
    )
    record(
        "7 productos declarados",
        f"{len(PRODUCTS)} producto(s) declarados, {len(fine)} a escala de edificio, "
        f"{sum(len(product.unknowns) for product in PRODUCTS)} incógnitas pendientes; "
        "ninguno por OGC y ninguno verificado",
    )


ABSENT: ninguna de las 59 capas anunciadas menciona infrarrojo, falso color ni índice espectral — 10 términos buscados en nombre, título y descripción
3 bandas de color + alfa en la imagen de 2023 a 2048 px (md5 14b2bfb2)


**Consecuencia para los indicadores de vegetación**

| | |
|---|---|
| 🛰️ Bandas que llegan | 3 de color + alfa — sin infrarrojo |
| 🌿 NDVI por esta vía | **descartado**: la ortofoto servida no trae infrarrojo |
| 🔁 Lo que queda por edificio, aquí | sustitutos RGB (exceso de verde, VARI): confunden plantas con cubiertas verdes y con suelo en sombra, así que un positivo no es vegetación hasta verificarlo a ojo |
| 🏠 Píxeles por edificio en la ortofoto | 1.600 px² a 0.25 m/px (medidos), para 10 m de lado |

Sin infrarrojo, la vegetación sobre una cubierta sólo se puede insinuar con color visible. Eso no invalida la detección de abandono: la desplaza hacia el cambio estructural entre épocas, que es lo que sí mide esta resolución.

,producto,bandas,m/px declarados,px² por edificio (peor caso),años,cadencia,formato,acceso,incógnitas
0,PNOA false-colour infrared orthophoto,NIR-R-G composite,0.15–0.25,1600.00,"annual, latest declared 2024","annual, national coverage","COG, cut into MTN25 sheets",download catalogue; no OGC service found,3
1,NDVI,"index, from red and near infrared",10,1.00,2018-2025,"bimonthly, one mosaic per autonomous community",COG,download catalogue; no OGC service advertised,1
2,NDWI,"index, from green and near infrared",10,1.00,2018-2025,"semiannual, one winter and one summer mosaic",COG,download catalogue; no OGC service advertised,1
3,Landsat historical mosaics,natural colour and false-colour infrared,15–30,0.11,1986 onwards,about one per decade,not declared on the page read,download catalogue,2


PENDIENTE: PNOA false-colour infrared orthophoto — whether this extent falls in a lot, and at which of the declared resolutions
PENDIENTE: PNOA false-colour infrared orthophoto — which band of the delivered file carries the near infrared
PENDIENTE: PNOA false-colour infrared orthophoto — whether a stable direct URL allows reading one bounding box by HTTP range instead of a whole sheet
PENDIENTE: NDVI — whether one bounding box can be read without the whole community mosaic
PENDIENTE: NDWI — whether one bounding box can be read without the whole community mosaic
PENDIENTE: Landsat historical mosaics — which decades cover this extent
PENDIENTE: Landsat historical mosaics — whether the false-colour composite carries a usable infrared band


**Lo que decide si un producto sirve: píxeles por edificio**

| | |
|---|---|
| 🏠 En la ortofoto ya medida | 1.600 px² a 0.25 m/px (medidos) |
| 🔍 Para juzgar un tejado | hacen falta cientos de px², y eso sólo lo dan los productos de decímetros: 1 de 4 declarados |
| 🌾 Para el entorno de la parcela | bastan decenas, así que ahí sí entran los de 10 m, y además aportan varias observaciones al año frente a una por época |
| 🚪 Cómo se accede | ninguno se sirve por OGC: todos son descarga, la vía que este instrumento no tiene |
| ⏳ Qué falta para que cuenten | 7 incógnitas, listadas arriba como PENDIENTE |

Declarado por sus catálogos y **no verificado aquí**: ninguna fila de `manifest.csv` lo respalda. El infrarrojo a escala de edificio existe en España; lo que no existe es por los servicios que este notebook interroga.

4 producto(s) declarados, 1 a escala de edificio, 7 incógnitas pendientes; ninguno por OGC y ninguno verificado


## 8 · El techo del servicio, medido, y la petición por teselas

En vivo, el extent entero se rechaza con estado 200 y un `ExceptionReport`: *Area of
extension out of limits*. No se quejó de ningún parámetro —la stored query
sustituida por `GetFeature` no quedó refutada—, sino del tamaño.

Así que esta celda mide el techo en vez de inventarlo: pide la misma forma a tamaños
decrecientes y para en la primera que conteste. Separa tres fracasos, porque sólo uno
habla del tamaño: el **rechazo** del servicio, que acota; el **timeout**, que dice que
seguía trabajando y deja seguir bajando; y el **servicio inalcanzable**,
que no mide nada y la detiene. Cada escalón deja su línea en el
manifiesto, y la cifra que sale va a la entrada del proveedor, nunca a una función.

Con ese techo el extent se parte en teselas que quepan debajo, pedidas una a una y
juntadas sin repetidos por identificador.

**Qué mirar:** la fila donde la raíz pasa de `ExceptionReport` a `FeatureCollection`, y
cuánto tardó. Debajo, cuántos edificios se repiten entre teselas.

In [11]:
# --- Sección 8: el techo del servicio, medido, y la petición por teselas ---
import pandas as pd

CADASTRE = eda.read_features(None)  # el valor "sin documento" hasta que haya respuesta
BUILDINGS: tuple[eda.FeatureRecord, ...] = ()
CEILING_KM2 = 0.0  # la mayor ventana que el servicio aceptó, medida en esta celda
AXIS_SWAPPED: bool | None = None


def ask_features(box: tuple[float, ...]) -> tuple[eda.Fetch, eda.FeatureSet]:
    """Pide las features de `box` y devuelve la petición y lo que trajo."""
    got = attempt(
        f"features {eda.area_km2(box):.3f} km²",
        lambda: eda.wfs_features(PROVIDER, "buildings", box, EXTENT_IN, OUT_DIR),
    )
    return got, eda.read_features(got.path)


def refusal(got: eda.Fetch, answer: eda.FeatureSet) -> str:
    """Por qué una petición no trajo una colección, o cadena vacía si la trajo."""
    return got.error or answer.exception


def verdict_of(got: eda.Fetch, answer: eda.FeatureSet, why: str) -> str:
    """Qué fue de una petición, que no es lo mismo que por qué falló.

    Son cuatro cosas distintas y sólo una de ellas habla del tamaño pedido: el
    rechazo del propio servicio, que llega como documento suyo. Un `timeout`
    dice que seguía trabajando, y una ventana menor es justo lo que sí puede
    contestar; no haber alcanzado el servicio no dice nada de nada.
    """
    if not why:
        return "aceptada"
    if answer.root and answer.exception:
        return "rechazada por el servicio"
    if got.timed_out:
        return "sin contestar a tiempo"
    return "servicio inalcanzable"


if not EXTENT_BBOX:
    absent("8 catastro", "sin extent no hay bbox que pedir al servicio de features")
else:
    panel(
        "La petición, tal como la define la entrada del proveedor",
        [
            (
                "🔌 Servicio y tipo",
                " · ".join(f"`{part}`" for part in PROVIDER.wfs_types["buildings"]),
            ),
            ("📨 Parámetros", ", ".join(f"`{key}`" for key in PROVIDER.wfs_params)),
            ("📐 Extent completo", f"{eda.area_km2(EXTENT_BBOX):.3f} km²"),
            (
                "🧱 Techo que declara la entrada",
                f"{PROVIDER.wfs_max_area_km2:g} km²"
                if PROVIDER.wfs_max_area_km2
                else "ninguno todavía: se mide aquí en vez de suponerlo",
            ),
        ],
        note=(
            "Los parámetros salen de la entrada del proveedor, no de esta celda. El "
            "servicio puede rechazar la petición por su tamaño sin quejarse de ninguno."
        ),
    )

    # 1 · El techo: la misma forma a tamaños decrecientes, hasta que conteste.
    rungs: list[dict[str, object]] = []
    accepted: eda.FeatureSet | None = None
    accepted_bbox: tuple[float, ...] = ()
    smallest_refused = 0.0
    smallest_slow = 0.0
    refused_count = 0
    slow_count = 0
    unreachable = False
    accepted_seconds = 0.0
    last_reason = ""  # el último fracaso, sea del tipo que sea
    refusal_words = ""  # y, aparte, lo que dijo el servicio al rechazar por tamaño
    for factor in eda.AREA_FACTORS:
        window = eda.scaled_window(EXTENT_BBOX, factor)
        got, answer = ask_features(window)
        why = refusal(got, answer)
        verdict = verdict_of(got, answer, why)
        rungs.append(
            {
                "ventana": f"{factor:.0%} del extent",
                "km²": round(eda.area_km2(window), 3),
                "veredicto": verdict,
                "estado": got.status,
                "segundos": round(got.elapsed_s, 1),
                "bytes": got.nbytes,
                "raíz": answer.root or "—",
                "features": len(answer.features),
                "respuesta": why or "colección de features",
            }
        )
        if verdict == "aceptada":
            accepted, accepted_bbox = answer, window
            accepted_seconds = got.elapsed_s
            break
        last_reason = why
        if verdict == "rechazada por el servicio":
            refused_count += 1
            smallest_refused = eda.area_km2(window)
            refusal_words = why  # sólo un rechazo suyo se puede citar como tal
        elif verdict == "sin contestar a tiempo":
            # El servicio está ahí y seguía trabajando: pedir menos es
            # exactamente lo que puede contestar, así que la escalera sigue.
            slow_count += 1
            smallest_slow = eda.area_km2(window)
        else:
            # Sin documento del servicio que leer, el rechazo no es suyo: es de
            # la red o de un intermediario, y una ventana menor no lo acercaría.
            unreachable = True
            break
    display(pd.DataFrame(rungs))

    if accepted is None and unreachable:
        absent(
            "8 techo",
            f"no se alcanzó el servicio ({last_reason}): su techo no se puede medir "
            "desde aquí, y pedir ventanas menores no lo acercaría",
        )
    elif accepted is None:
        absent(
            "8 techo",
            f"ninguna de las {len(rungs)} ventanas contestó — "
            f"{refused_count} rechazada(s) por tamaño"
            + (
                f", la menor de {smallest_refused:.3f} km² ({refusal_words})"
                if refused_count
                else ""
            )
            + f"; {slow_count} sin contestar en {PROVIDER.wfs_timeout_s:g} s"
            + (f", la menor de {smallest_slow:.3f} km²" if slow_count else "")
            + f" — último motivo: {last_reason}",
        )
    if accepted is None:
        absent("8 catastro", "sin una sola ventana aceptada no hay edificios que medir")
    else:
        CEILING_KM2 = eda.area_km2(accepted_bbox)
        bracket = (
            f"entre {CEILING_KM2:.3f} y {smallest_refused:.3f} km²"
            if smallest_refused
            else f"sin alcanzar: aceptó el extent entero, {CEILING_KM2:.3f} km²"
        )
        if slow_count:  # una ventana lenta no acota nada, pero sí avisa
            bracket += (
                f", con {slow_count} ventana(s) sin contestar a tiempo por el medio"
            )
        panel(
            "El techo del servicio, acotado por sus propias respuestas",
            [
                ("✅ Mayor ventana aceptada", f"{CEILING_KM2:.3f} km²"),
                (
                    "❌ Menor ventana rechazada",
                    f"{smallest_refused:.3f} km²" if smallest_refused else "ninguna",
                ),
                ("📏 El techo está", bracket),
                (
                    "🗣️ Con sus palabras",
                    refusal_words or "no hubo rechazo que citar",
                ),
                (
                    "⏱️ Espera",
                    f"límite {PROVIDER.wfs_timeout_s:g} s · esta ventana tardó "
                    f"{accepted_seconds:.1f} s"
                    + (
                        f" · {slow_count} agotaron la espera"
                        if slow_count
                        else " · ninguna la agotó"
                    ),
                ),
                ("🧮 Peticiones gastadas en medirlo", f"{len(rungs)}"),
            ],
            note=(
                "Acotado, no exacto: para estrecharlo más harían falta más peticiones, "
                "y lo que el pipeline necesita es un tamaño que el servicio acepte."
            ),
        )
        print(f"para la entrada del proveedor: wfs_max_area_km2={CEILING_KM2:.3f}")
        if smallest_refused:
            ceiling_line = (
                f"techo del servicio {bracket}: acepta {CEILING_KM2:.3f} km² en "
                f"{accepted_seconds:.1f} s y rechaza {smallest_refused:.3f} km² "
                f"({refusal_words})"
            )
        else:
            ceiling_line = (
                f"techo del servicio sin alcanzar: aceptó el extent entero, "
                f"{CEILING_KM2:.3f} km², en la primera petición"
            )
        record("8 techo", ceiling_line)

        # 2 · Las teselas: el extent partido en trozos que el servicio acepta.
        tiles = eda.split_under(EXTENT_BBOX, CEILING_KM2)
        groups: list[tuple[eda.FeatureRecord, ...]] = []
        answers: list[eda.FeatureSet] = []
        tile_rows: list[dict[str, object]] = []
        served_as = ""
        truncated = 0
        undeclared = 0
        for index, tile in enumerate(tiles, 1):
            got, answer = ask_features(tile)
            why = refusal(got, answer)
            tile_verdict = verdict_of(got, answer, why)
            cut = 0 <= answer.number_returned < answer.number_matched
            silent = answer.number_matched < 0  # no declara cuántas encontró
            truncated += 1 if cut else 0
            undeclared += 1 if silent and not why else 0
            if why:
                print(
                    f"ABSENT: tesela {index} de {len(tiles)}, {tile_verdict} — {why}"
                )
            else:
                answers.append(answer)
                groups.append(answer.features)
                served_as = served_as or (got.path.suffix if got.path else "")
            tile_rows.append(
                {
                    "tesela": f"{index}/{len(tiles)}",
                    "km²": round(eda.area_km2(tile), 3),
                    "veredicto": tile_verdict,
                    "segundos": round(got.elapsed_s, 1),
                    "estado": got.status,
                    "bytes": got.nbytes,
                    "features": len(answer.features),
                    "declara devueltas": answer.number_returned,
                    "declara encontradas": answer.number_matched,
                    "truncada": "sí" if cut else ("sin declarar" if silent else "no"),
                    "md5": got.md5[:8],
                }
            )
        display(pd.DataFrame(tile_rows))

        # El formato se describe con la primera respuesta que traiga algo, y si
        # ninguna trae nada, con la ventana que el servicio aceptó al medir el techo.
        CADASTRE = next((reply for reply in answers if reply.features), accepted)
        merged, repeated = eda.merge_features(groups)
        AXIS_SWAPPED, axis_reason = eda.axis_order(merged, EXTENT_BBOX)
        # Un eje invertido se dibuja en el mar sin que nada parezca roto, así que la
        # geometría se coloca según dónde cae, no según la gramática de la petición.
        BUILDINGS = eda.swap_axes(merged) if AXIS_SWAPPED else merged
        MEASURED["cadastre_request"] = {
            "extent_km2": eda.area_km2(EXTENT_BBOX),
            "ceiling_accepted_km2": CEILING_KM2,
            "ceiling_refused_km2": smallest_refused,
            "refusal": refusal_words,
            "last_reason": last_reason,
            "slow_windows": slow_count,
            "smallest_slow_km2": smallest_slow,
            "timeout_s": PROVIDER.wfs_timeout_s,
            "accepted_seconds": accepted_seconds,
            "probe": rungs,
            "tiles": tile_rows,
            "features": len(BUILDINGS),
            "repeated_between_tiles": repeated,
            "truncated_tiles": truncated,
            "tiles_without_declared_counts": undeclared,
            "crs": CADASTRE.crs,
            "srs_name": CADASTRE.srs_name,
            "axis_swapped": AXIS_SWAPPED,
        }
        if not BUILDINGS:
            absent(
                "8 catastro",
                f"{len(groups)} de {len(tiles)} teselas contestaron, y ninguna trae una "
                "sola feature: el servicio no publica edificios sobre este extent",
            )
        else:
            panel(
                "Los edificios del extent, juntando las teselas",
                [
                    (
                        "📦 Formato",
                        f"`{CADASTRE.root}`"
                        + (f" servido como `{served_as}`" if served_as else ""),
                    ),
                    (
                        "🏷️ Tipo(s) de feature",
                        ", ".join(f"`{name}`" for name in CADASTRE.type_names) or "—",
                    ),
                    (
                        "🗺️ CRS de la respuesta",
                        f"`{CADASTRE.srs_name or 'sin declarar'}` → `{CADASTRE.crs}`"
                        f" · pedido en `{EXTENT_IN}`",
                    ),
                    ("🧩 Teselas que contestaron", f"{len(groups)} de {len(tiles)}"),
                    (
                        "🏠 Edificios distintos",
                        f"{len(BUILDINGS)} · {repeated} repetidos entre teselas",
                    ),
                    (
                        "✂️ ¿Respuestas completas?",
                        f"{truncated} tesela(s) truncada(s): devolvieron menos de lo "
                        "que dijeron haber encontrado"
                        if truncated
                        else (
                            f"no se puede saber: {undeclared} tesela(s) no declaran "
                            "cuántas features encontraron, y el servicio no anuncia "
                            "paginación"
                            if undeclared
                            else "sí: cada tesela devolvió lo que dijo haber encontrado"
                        ),
                    ),
                    ("🧭 Orden de los ejes", axis_reason),
                ],
                note=(
                    "Cada tesela deja su línea en el manifiesto y su cuerpo en "
                    "`out/vectors/`: la cifra de arriba se puede rehacer desde el disco."
                ),
            )
            if AXIS_SWAPPED is None:
                print(f"ABSENT: posición de las huellas — {axis_reason}")
            record(
                "8 catastro",
                f"catastro: {len(BUILDINGS)} edificios distintos de tipo "
                f"{'/'.join(CADASTRE.type_names) or 'sin nombre'} en {CADASTRE.crs}, "
                f"{len(groups)}/{len(tiles)} teselas bajo un techo de "
                f"{CEILING_KM2:.3f} km², {repeated} repetidos entre teselas",
            )

**La petición, tal como la define la entrada del proveedor**

| | |
|---|---|
| 🔌 Servicio y tipo | `buildings_wfs` · `bu:Building` |
| 📨 Parámetros | `service`, `version`, `request`, `typenames`, `namespaces`, `srsname`, `bbox` |
| 📐 Extent completo | 1.204 km² |
| 🧱 Techo que declara la entrada | 0.975 km² |

Los parámetros salen de la entrada del proveedor, no de esta celda. El servicio puede rechazar la petición por su tamaño sin quejarse de ninguno.

,ventana,km²,veredicto,estado,segundos,bytes,raíz,features,respuesta
0,100% del extent,1.204,rechazada por el servicio,200,9.3,300,ExceptionReport,0,service exception: Area of extension out of li...
1,90% del extent,0.975,aceptada,200,69.6,2617533,FeatureCollection,545,colección de features


**El techo del servicio, acotado por sus propias respuestas**

| | |
|---|---|
| ✅ Mayor ventana aceptada | 0.975 km² |
| ❌ Menor ventana rechazada | 1.204 km² |
| 📏 El techo está | entre 0.975 y 1.204 km² |
| 🗣️ Con sus palabras | service exception: Area of extension out of limits |
| ⏱️ Espera | límite 120 s · esta ventana tardó 69.6 s · ninguna la agotó |
| 🧮 Peticiones gastadas en medirlo | 2 |

Acotado, no exacto: para estrecharlo más harían falta más peticiones, y lo que el pipeline necesita es un tamaño que el servicio acepte.

para la entrada del proveedor: wfs_max_area_km2=0.975
techo del servicio entre 0.975 y 1.204 km²: acepta 0.975 km² en 69.6 s y rechaza 1.204 km² (service exception: Area of extension out of limits)


,tesela,km²,veredicto,segundos,estado,bytes,features,declara devueltas,declara encontradas,truncada,md5
0,1/2,0.602,aceptada,86.8,200,2682229,562,-1,-1,sin declarar,b71702e0
1,2/2,0.602,aceptada,24.0,200,1159461,228,-1,-1,sin declarar,9e87c55f


**Los edificios del extent, juntando las teselas**

| | |
|---|---|
| 📦 Formato | `FeatureCollection` servido como `.xml` |
| 🏷️ Tipo(s) de feature | `Building` |
| 🗺️ CRS de la respuesta | `urn:ogc:def:crs:EPSG::25830` → `EPSG:25830` · pedido en `EPSG:25830` |
| 🧩 Teselas que contestaron | 2 de 2 |
| 🏠 Edificios distintos | 787 · 3 repetidos entre teselas |
| ✂️ ¿Respuestas completas? | no se puede saber: 2 tesela(s) no declaran cuántas features encontraron, y el servicio no anuncia paginación |
| 🧭 Orden de los ejes | as requested: the geometry falls inside the extent |

Cada tesela deja su línea en el manifiesto y su cuerpo en `out/vectors/`: la cifra de arriba se puede rehacer desde el disco.

catastro: 787 edificios distintos de tipo Building en EPSG:25830, 2/2 teselas bajo un techo de 0.975 km², 3 repetidos entre teselas


### 8.1 · Qué trae cada edificio, y si eso puede etiquetar

Con los edificios ya en memoria, esta celda mira qué viene con cada uno: primero los
atributos que la entrada del proveedor afirma, uno por uno y con su nombre real, y
después el inventario completo, que incluye lo que llega sin que nadie lo hubiera
previsto.

Dos cifras deciden si el catastro puede servir de etiqueta débil: cuántos valores
distintos toma el atributo de condición sobre este extent, y si llega un año de
construcción. Un atributo que existe pero sale uniforme no etiqueta nada. Qué valor
significa abandono no se decide aquí: eso es ADR-004.

Al final, la superficie de cada huella con los patios restados. Esa cifra retira el
supuesto `BUILDING_SIDE_M` que la sección 7 arrastra para contar píxeles por edificio,
y lo sustituye por el tamaño que tienen los edificios de este extent.

**Qué mirar:** los valores distintos de la condición. Dos o más y el etiquetado de
EXP-001 cambia de forma; uno solo, o ninguno, y sigue siendo manual.

In [12]:
# --- Sección 8.1: qué trae cada edificio, y si eso puede etiquetar ---
import pandas as pd


def miles(value: float) -> str:
    """Separador de miles a la española, como en la sección 7: 1.600."""
    return f"{value:,.0f}".replace(",", ".")


BUILT = globals().get("BUILDINGS", ())  # los edificios que juntó la sección 8
REPLY = globals().get("CADASTRE", None)  # y la respuesta que describe su formato

if not BUILT:
    absent("8.1 atributos", "sin edificios en memoria: ejecuta antes la sección 8")
else:
    total = len(BUILT)
    reply_crs = REPLY.crs if REPLY is not None else ""

    # Lo que la entrada afirma, uno por uno y con su nombre real.
    coverage = eda.attribute_coverage(BUILT)

    def filled(name: str) -> int:
        """Cuántos edificios traen ese campo **con un valor dentro**.

        Publicar el nombre y dejarlo vacío no es lo mismo que publicarlo: el
        primero se cuenta en `coverage` y no sirve para nada aguas abajo.
        """
        return sum(1 for building in BUILT if building.values.get(name))
    claimed = [
        {
            "atributo": logical,
            "nombre real": name,
            "presente": "sí" if name in coverage else "no",
            "edificios que lo traen": coverage.get(name, 0),
            "con valor": filled(name),
            "ejemplo": next((b.values[name] for b in BUILT if b.values.get(name)), ""),
        }
        for logical, name in PROVIDER.attributes.items()
    ]
    display(pd.DataFrame(claimed))
    for row in claimed:
        if row["presente"] == "no":
            print(
                f"ABSENT: atributo '{row['atributo']}' — ningún edificio publica "
                f"'{row['nombre real']}'"
            )
        elif not row["con valor"]:
            print(
                f"ABSENT: atributo '{row['atributo']}' — '{row['nombre real']}' se "
                f"publica en los {total} edificios y llega vacío en todos: está el "
                "nombre, no el dato"
            )

    # Y el inventario completo: también lo que llega sin que nadie lo afirmara.
    display(
        pd.DataFrame(
            [
                {
                    "campo": name,
                    "edificios": count,
                    "con valor": filled(name),
                    "cobertura": f"{count / total:.0%}",
                    "ejemplo": next(
                        (b.values[name] for b in BUILT if b.values.get(name)),
                        "(siempre vacío)",
                    ),
                }
                for name, count in coverage.items()
            ]
        )
    )

    # Condición y antigüedad: las dos que deciden si esto puede etiquetar.
    condition_name = PROVIDER.attributes.get("condition", "")
    condition_counts = eda.value_counts(BUILT, condition_name) if condition_name else {}
    distinct = [value for value in condition_counts if value]
    if not condition_counts:
        absent(
            "8.1 etiqueta débil",
            f"ningún edificio trae '{condition_name or 'atributo de condición'}', "
            "así que el etiquetado de EXP-001 sigue siendo manual",
        )
    else:
        display(
            pd.DataFrame(
                [
                    {
                        "condición": value or "(vacío)",
                        "edificios": count,
                        "parte": f"{count / total:.0%}",
                    }
                    for value, count in condition_counts.items()
                ]
            )
        )
        if len(distinct) < 2:
            absent(
                "8.1 etiqueta débil",
                f"'{condition_name}' llega uniforme sobre el extent "
                f"({distinct[0] if distinct else 'sin valor'} en {total} edificios): "
                "no discrimina, y el etiquetado sigue siendo manual",
            )
        else:
            record(
                "8.1 etiqueta débil",
                f"'{condition_name}' toma {len(distinct)} valores distintos sobre "
                f"{total} edificios: hay con qué etiquetar débilmente, y qué valor "
                "significa abandono lo decide ADR-004, no esta celda",
            )

    year_name = PROVIDER.attributes.get("year", "")
    ages = eda.value_counts(BUILT, year_name) if year_name else {}

    def decade(value: str) -> str:
        """La década de un valor que empieza por un año, o su ausencia."""
        head = value[:4]
        return f"{head[:3]}0" if head.isdigit() else "sin año legible"

    decades: dict[str, int] = {}
    for value, count in ages.items():
        decades[decade(value)] = decades.get(decade(value), 0) + count
    if not ages:
        absent(
            "8.1 antigüedad",
            f"ningún edificio trae '{year_name or 'atributo de año'}': sin antigüedad "
            "no hay forma de separar la ruina antigua de la obra reciente",
        )
    else:
        display(
            pd.DataFrame(
                [
                    {"década": key, "edificios": count}
                    for key, count in sorted(decades.items())
                ]
            )
        )
        legible = sum(count for key, count in decades.items() if key.isdigit())
        carried = sum(ages.values())  # edificios, no valores distintos
        record(
            "8.1 antigüedad",
            f"'{year_name}' llega en {carried} de {total} edificios, {legible} de "
            f"ellos con año legible; {len(ages)} fechas distintas repartidas en "
            f"{len([key for key in decades if key.isdigit()])} décadas",
        )

    # El tamaño del edificio, que hasta aquí era un supuesto declarado.
    areas = sorted(eda.footprint_area(b) for b in BUILT if b.exterior)
    metres = reply_crs == EXTENT_IN  # proyectado y el mismo que se pidió: metros
    unit = "m²" if metres else f"unidades² de {reply_crs or 'su CRS'}"
    footprint: dict[str, object] = {}
    if not areas:
        absent(
            "8.1 huellas",
            "los edificios llegan sin geometría: no hay superficie que medir ni "
            "huellas que dibujar",
        )
    else:
        middle = areas[len(areas) // 2]
        side = middle**0.5
        parts_each = [len(building.exterior) for building in BUILT if building.exterior]
        aggregates = sum(1 for count in parts_each if count > 1)
        footprint = {
            "count": len(areas),
            "median": middle,
            "p25": areas[len(areas) // 4],
            "p75": areas[(3 * len(areas)) // 4],
            "min": areas[0],
            "max": areas[-1],
            "unit": unit,
        }
        panel(
            "El tamaño del edificio, medido en vez de supuesto",
            [
                ("🏠 Huellas con geometría", f"{len(areas)} de {total}"),
                (
                    "📐 Superficie mediana",
                    f"{miles(middle)} {unit} · cuartiles "
                    f"{miles(footprint['p25'])}–{miles(footprint['p75'])}",
                ),
                (
                    "📏 Lado equivalente",
                    f"{side:.1f} m de cuadrado" if metres else "sin metros, sin lado",
                ),
                (
                    "🧮 Supuesto de la sección 2",
                    f"BUILDING_SIDE_M = {BUILDING_SIDE_M:g} m — "
                    + (f"medido {side:.1f} m" if metres else "no comparable aquí"),
                ),
                (
                    "🖼️ Píxeles por edificio mediano",
                    f"{miles(middle / GSD_M**2)} px² a {GSD_M:g} m/px"
                    if metres
                    else "—",
                ),
                (
                    "🧩 Huellas de varias partes",
                    f"{aggregates} de {len(areas)} · la mayor junta "
                    f"{max(parts_each)} partes y {miles(areas[-1])} {unit}, así que "
                    "una feature del catastro no es siempre un edificio",
                ),
            ],
            note=(
                "La superficie sale de los anillos exteriores con los patios restados, "
                "y sólo es una medida en metros cuando la respuesta llega en el CRS "
                "proyectado que se pidió."
            ),
        )

    MEASURED["cadastre_attributes"] = {
        "buildings": total,
        "attributes": coverage,
        "claimed": claimed,
        "condition": {"name": condition_name, "counts": condition_counts},
        "age": {"name": year_name, "decades": decades},
        "footprint": footprint,
        "fields_without_values": [name for name in coverage if not filled(name)],
    }
    record(
        "8.1 atributos",
        f"atributos: {len(coverage)} campos distintos sobre {total} edificios, "
        f"{sum(1 for row in claimed if row['presente'] == 'sí')} de {len(claimed)} "
        "afirmados por la entrada presentes con su nombre real",
    )

,atributo,nombre real,presente,edificios que lo traen,con valor,ejemplo
0,identifier,localId,sí,787,787,9135702VK6893S
1,condition,conditionOfConstruction,sí,787,787,functional
2,year,beginning,sí,787,787,2003-01-01T00:00:00
3,use,currentUse,sí,787,787,1_residential
4,floors,numberOfFloorsAboveGround,sí,787,0,


ABSENT: atributo 'floors' — 'numberOfFloorsAboveGround' se publica en los 787 edificios y llega vacío en todos: está el nombre, no el dato


,campo,edificios,con valor,cobertura,ejemplo
0,addresses,787,787,100%,wfsAD.aspx?service=wfs&version=2&request=GetFe...
1,beginLifespanVersion,787,787,100%,2008-03-11T00:00:00
2,beginning,787,787,100%,2003-01-01T00:00:00
3,cadastralParcels,787,787,100%,wfsCP.aspx?service=wfs&version=2&request=GetFe...
4,conditionOfConstruction,787,787,100%,functional
5,currentUse,787,787,100%,1_residential
6,documentLink,787,787,100%,http://ovc.catastro.meh.es/OVCServWeb/OVCWcfLi...
7,end,787,787,100%,2003-01-01T00:00:00
8,endLifespanVersion,787,0,100%,(siempre vacío)
9,format,787,787,100%,jpeg


,condición,edificios,parte
0,functional,787,100%


ABSENT: 'conditionOfConstruction' llega uniforme sobre el extent (functional en 787 edificios): no discrimina, y el etiquetado sigue siendo manual


,década,edificios
0,1950,2
1,1960,2
2,1970,6
3,1980,30
4,1990,134
5,2000,593
6,2010,2
7,2020,18


'beginning' llega en 787 de 787 edificios, 787 de ellos con año legible; 36 fechas distintas repartidas en 8 décadas


**El tamaño del edificio, medido en vez de supuesto**

| | |
|---|---|
| 🏠 Huellas con geometría | 787 de 787 |
| 📐 Superficie mediana | 71 m² · cuartiles 64–78 |
| 📏 Lado equivalente | 8.4 m de cuadrado |
| 🧮 Supuesto de la sección 2 | BUILDING_SIDE_M = 10 m — medido 8.4 m |
| 🖼️ Píxeles por edificio mediano | 1.134 px² a 0.25 m/px |
| 🧩 Huellas de varias partes | 30 de 787 · la mayor junta 113 partes y 61.439 m², así que una feature del catastro no es siempre un edificio |

La superficie sale de los anillos exteriores con los patios restados, y sólo es una medida en metros cuando la respuesta llega en el CRS proyectado que se pidió.

atributos: 26 campos distintos sobre 787 edificios, 5 de 5 afirmados por la entrada presentes con su nombre real


### 8.2 · Las huellas, sobre la ortofoto más reciente

Las secciones 8 y 8.1 los contaron y midieron; ésta los pone donde están. Pide **una** imagen de la época
más reciente que la sección 5 encontró con píxeles y dibuja encima el contorno de cada
huella, con los patios en trazo discontinuo.

Es la comprobación que ninguna tabla da: si los contornos se apoyan en los tejados,
entonces el CRS, el orden de los ejes y la extensión coinciden de verdad; si caen
desplazados, hay una cifra arriba que no significa lo que parece.

La figura se guarda en `out/figs/` y se muestra desde disco, para no meter otro megabyte
de base64 dentro del notebook. Si la sección 8 no se ha ejecutado, aquí no hay nada
que dibujar y se dice en una línea.

**Qué mirar:** si los contornos se apoyan en los tejados. Un desplazamiento constante
apunta al CRS; un revoltijo, al orden de los ejes.

In [13]:
# --- Sección 8.2: las huellas, sobre la ortofoto más reciente que hay ---
import matplotlib.pyplot as plt

FOOTPRINTS = globals().get("BUILDINGS", ())  # lo que leyó la sección 8, si se ejecutó
if not FOOTPRINTS:
    absent("8.2 huellas vistas", "sin huellas en memoria: ejecuta antes la sección 8")
elif not EXTENT_BBOX:
    absent("8.2 huellas vistas", "sin extent no hay marco sobre el que dibujarlas")
else:
    min_x, min_y, max_x, max_y = EXTENT_BBOX
    epochs = MEASURED.get("epochs", {})
    years = [int(year) for year in epochs.get("present_years", [])]
    figure, axis = plt.subplots(figsize=(9, 9 * (max_y - min_y) / (max_x - min_x)))
    background = ""
    BACKGROUND_PATH = None  # la 8.3 reutiliza esta imagen en vez de volver a pedirla

    if not years:
        print("ABSENT: fondo — la sección 5 no encontró ninguna época con imagen")
    else:
        latest = max(years)  # la más reciente MEDIDA, no la más reciente anunciada
        shot = attempt(
            f"ortofoto {latest}",
            lambda: eda.wms_map(
                PROVIDER,
                PROVIDER.epoch_layer_pattern.format(year=latest),
                EXTENT_BBOX,
                EXTENT_IN,
                (REQUEST_PX, REQUEST_PX),
                OUT_DIR,
            ),
        )
        if shot.path is None:
            print(f"ABSENT: fondo — {shot.error or 'la época más reciente no dio imagen'}")
        else:
            axis.imshow(plt.imread(shot.path), extent=(min_x, max_x, min_y, max_y))
            background = f"{latest} · {REQUEST_PX} px · md5 {shot.md5[:8]}"
            BACKGROUND_PATH = shot.path

    holes = 0
    for building in FOOTPRINTS:
        for ring in building.exterior:
            axis.plot(
                [x for x, _ in ring], [y for _, y in ring], linewidth=0.7, color="#ffe14d"
            )
        for ring in building.interior:
            holes += 1
            axis.plot(
                [x for x, _ in ring],
                [y for _, y in ring],
                linewidth=0.6,
                linestyle="--",
                color="#4dd2ff",
            )

    def touches_extent(building: eda.FeatureRecord) -> bool:
        """Si la huella corta el extent pedido, que es lo que el filtro prometía."""
        box = eda.rings_envelope([building])
        return box is not None and (
            box[0] <= max_x and min_x <= box[2] and box[1] <= max_y and min_y <= box[3]
        )

    inside = sum(1 for building in FOOTPRINTS if touches_extent(building))
    axis.set_xlim(min_x, max_x)
    axis.set_ylim(min_y, max_y)
    axis.set_aspect("equal")
    axis.set_title(
        f"{len(FOOTPRINTS)} huellas catastrales · fondo: {background or 'ausente'}",
        fontsize=10,
    )
    axis.tick_params(labelsize=7)
    figure.tight_layout()
    FOOTPRINT_FIG = OUT_DIR / "figs" / "footprints.png"
    figure.savefig(FOOTPRINT_FIG, dpi=200, bbox_inches="tight")
    plt.close(figure)
    # Referencia, no bytes: `Image(filename=…)` incrustaría el PNG en el output.
    display(Markdown(f"![Huellas catastrales]({FOOTPRINT_FIG.relative_to(HERE)})"))

    if inside < len(FOOTPRINTS):
        print(
            f"ABSENT: {len(FOOTPRINTS) - inside} huella(s) fuera del extent pedido — "
            "el filtro espacial no acotó lo que prometía, o el CRS no es el que dice"
        )
    panel(
        "Las huellas, contra el extent que se pidió",
        [
            ("🏠 Huellas dibujadas", f"{len(FOOTPRINTS)}"),
            ("🎯 Que cortan el extent", f"{inside} de {len(FOOTPRINTS)}"),
            ("🕳️ Patios dibujados", f"{holes}"),
            ("🛰️ Fondo", background or "ausente: sólo los contornos"),
            (
                "🔍 Suelo por píxel del fondo",
                f"{(max_x - min_x) / REQUEST_PX:.2f} m/px — vista de conjunto: "
                "enseña un desplazamiento, no un error de centímetros",
            ),
            ("💾 Figura", f"`{FOOTPRINT_FIG.relative_to(HERE)}`"),
        ],
        note=(
            "Fuera de git, porque se vuelve a dibujar. El fondo es una petición más, "
            "con su línea en el manifiesto, y la del contorno es la de la sección 8."
        ),
    )
    record(
        "8.2 huellas vistas",
        f"{len(FOOTPRINTS)} huellas dibujadas ({holes} patios), {inside} cortan el "
        f"extent, sobre {background or 'ningún fondo: la ortofoto no llegó'}",
    )

![Huellas catastrales](out/figs/footprints.png)

ABSENT: 10 huella(s) fuera del extent pedido — el filtro espacial no acotó lo que prometía, o el CRS no es el que dice


**Las huellas, contra el extent que se pidió**

| | |
|---|---|
| 🏠 Huellas dibujadas | 787 |
| 🎯 Que cortan el extent | 777 de 787 |
| 🕳️ Patios dibujados | 31 |
| 🛰️ Fondo | 2023 · 512 px · md5 f9c4309b |
| 🔍 Suelo por píxel del fondo | 2.62 m/px — vista de conjunto: enseña un desplazamiento, no un error de centímetros |
| 💾 Figura | `out/figs/footprints.png` |

Fuera de git, porque se vuelve a dibujar. El fondo es una petición más, con su línea en el manifiesto, y la del contorno es la de la sección 8.

787 huellas dibujadas (31 patios), 777 cortan el extent, sobre 2023 · 512 px · md5 f9c4309b


### 8.3 · Los demás tipos de feature, y los servicios hermanos

El catastro no publica sólo edificios. Su WFS sirve además las **partes** de cada
edificio y las **otras construcciones** —piscinas, silos, marquesinas—, y las respuestas
de la sección 8 enlazan dos servicios hermanos: **direcciones** y **parcelas**. Aquí se le pide a cada uno lo mismo que al primero, sobre las
mismas teselas, para dejar comprobado a qué datos se puede llegar de verdad.

Esto importa más allá del inventario: el número de plantas llega vacío en los edificios y
en el modelo INSPIRE vive en las partes; y la parcela es el terreno alrededor del
edificio, que es donde se ve la vegetación. Son cuatro tipos por tesela: ocho peticiones,
varios minutos.

Al final, la muestra: las huellas nuevas dibujadas sobre la misma ortofoto de la 8.2, y
una tabla con ejemplos concretos de las otras construcciones.

**Qué mirar:** la columna `features` por tipo. Un cero con motivo dice que ese dato no se
pide así; un número dice que sí, y entonces sus campos categóricos están debajo.

In [14]:
# --- Sección 8.3: los demás tipos de feature, sobre las mismas teselas ---
import matplotlib.pyplot as plt
import pandas as pd

CATEGORICAL_MAX = 8  # cuántos valores distintos hacen de un campo una categoría
TILES = globals().get("tiles", ())  # las teselas que el techo de la sección 8 fijó
BY_TYPE: dict[str, tuple[eda.FeatureRecord, ...]] = {}

if not TILES:
    absent("8.3 otros tipos", "sin las teselas de la sección 8 no hay dónde pedirlos")
else:
    rows: list[dict[str, object]] = []
    for kind, (service, type_name) in PROVIDER.wfs_types.items():
        reasons: list[str] = []
        seconds = 0.0
        if kind == "buildings":  # ya pedido arriba: no se vuelve a gastar una petición
            merged, repeated = globals().get("BUILDINGS", ()), 0
        else:
            groups: list[tuple[eda.FeatureRecord, ...]] = []
            for tile in TILES:
                got = attempt(
                    f"{kind} {eda.area_km2(tile):.3f} km²",
                    lambda k=kind, t=tile: eda.wfs_features(
                        PROVIDER, k, t, EXTENT_IN, OUT_DIR
                    ),
                )
                answer = eda.read_features(got.path)
                why = refusal(got, answer)
                seconds += got.elapsed_s
                if why:
                    reasons.append(f"{verdict_of(got, answer, why)} — {why}")
                else:
                    groups.append(answer.features)
            merged, repeated = eda.merge_features(groups)
            swapped, _ = eda.axis_order(merged, EXTENT_BBOX)
            merged = eda.swap_axes(merged) if swapped else merged
        BY_TYPE[kind] = merged
        coverage = eda.attribute_coverage(merged)
        rows.append(
            {
                "tipo": kind,
                "nombre": type_name,
                "servicio": service,
                "features": len(merged),
                "con geometría": sum(1 for one in merged if one.exterior),
                "repetidas": repeated,
                "campos": len(coverage),
                "campos con valor": sum(
                    1
                    for name in coverage
                    if any(one.values.get(name) for one in merged)
                ),
                "segundos": round(seconds, 1),
                "respuesta": reasons[0][:70] if reasons else "colección de features",
            }
        )
        if reasons:
            print(f"ABSENT: {kind} (`{type_name}` en {service}) — {reasons[0]}")
    display(pd.DataFrame(rows))

    # Los campos categóricos de cada tipo, que es donde podría vivir una etiqueta.
    # No se nombra ninguno: se descubren contando cuántos valores distintos toman.
    for kind, features in BY_TYPE.items():
        if not features:
            continue
        pieces: list[str] = []
        for name in eda.attribute_coverage(features):
            values = eda.value_counts(features, name)
            distinct = [value for value in values if value]
            if 1 <= len(distinct) <= CATEGORICAL_MAX:
                top = ", ".join(f"{value} ({values[value]})" for value in distinct[:4])
                pieces.append(f"`{name}`: {top}")
        display(
            Markdown(
                f"**{kind}** ({len(features)}) — "
                + (" · ".join(pieces[:5]) or "ningún campo categórico")
            )
        )

    # La muestra: qué son esas otras construcciones, una por una.
    NEW = {
        kind: features
        for kind, features in BY_TYPE.items()
        if kind != "buildings" and features
    }
    for kind, features in NEW.items():
        shown = sorted(features, key=eda.footprint_area, reverse=True)[:8]
        if not any(one.exterior for one in shown):
            continue
        display(
            pd.DataFrame(
                [
                    {
                        "tipo": kind,
                        "id": one.fid.rpartition(".")[2],
                        "m²": round(eda.footprint_area(one), 1),
                        "partes": len(one.exterior),
                        **{
                            name: one.values[name]
                            for name in list(one.values)[:4]
                            if one.values.get(name)
                        },
                    }
                    for one in shown
                ]
            )
        )

    # Y dibujadas sobre la misma ortofoto que la sección 8.2, sin pedirla de nuevo.
    drawable = {
        kind: features
        for kind, features in NEW.items()
        if any(one.exterior for one in features)
    }
    if not drawable:
        absent("8.3 muestra", "ningún tipo nuevo trajo geometría que dibujar")
    else:
        min_x, min_y, max_x, max_y = EXTENT_BBOX
        figure, axis = plt.subplots(figsize=(9, 9 * (max_y - min_y) / (max_x - min_x)))
        shot = globals().get("BACKGROUND_PATH")
        if shot is not None:
            axis.imshow(plt.imread(shot), extent=(min_x, max_x, min_y, max_y))
        else:
            print("ABSENT: fondo — la sección 8.2 no dejó ninguna imagen que reutilizar")
        colours = ["#ffe14d", "#4dd2ff", "#ff7ad9", "#9dff6e"]
        for (kind, features), colour in zip(drawable.items(), colours, strict=False):
            drawn = 0
            for one in features:
                for ring in one.exterior:
                    axis.plot(
                        [x for x, _ in ring],
                        [y for _, y in ring],
                        color=colour,
                        linewidth=0.7,
                        marker="." if len(ring) == 1 else "",
                        markersize=2,
                        label=f"{kind} ({len(features)})" if not drawn else "",
                    )
                    drawn += 1
        axis.set_xlim(min_x, max_x)
        axis.set_ylim(min_y, max_y)
        axis.set_aspect("equal")
        axis.legend(loc="upper right", fontsize=7, framealpha=0.8)
        axis.set_title("Los tipos de feature que la sección 8 no pidió", fontsize=10)
        axis.tick_params(labelsize=7)
        figure.tight_layout()
        TYPES_FIG = OUT_DIR / "figs" / "feature-types.png"
        figure.savefig(TYPES_FIG, dpi=200, bbox_inches="tight")
        plt.close(figure)
        display(Markdown(f"![Otros tipos de feature]({TYPES_FIG.relative_to(HERE)})"))

    answered = [kind for kind, features in BY_TYPE.items() if features]
    record(
        "8.3 otros tipos",
        f"tipos de feature: contestaron {len(answered)} de {len(PROVIDER.wfs_types)} "
        f"({', '.join(f'{k}={len(BY_TYPE[k])}' for k in answered)}); "
        f"{len(PROVIDER.wfs_types) - len(answered)} sin datos sobre el extent",
    )
    MEASURED["feature_types"] = {
        "probed": rows,
        "counts": {kind: len(features) for kind, features in BY_TYPE.items()},
    }

,tipo,nombre,servicio,features,con geometría,repetidas,campos,campos con valor,segundos,respuesta
0,buildings,bu:Building,buildings_wfs,787,787,0,26,23,0.0,colección de features
1,building_parts,bu:BuildingPart,buildings_wfs,6355,6355,481,12,11,47.7,colección de features
2,other_constructions,bu:OtherConstruction,buildings_wfs,124,124,23,9,8,32.9,colección de features
3,addresses,ad:Address,addresses_wfs,782,744,4,26,18,4.1,colección de features
4,parcels,cp:CadastralParcel,parcels_wfs,826,826,4,7,6,18.7,colección de features


**buildings** (787) — `conditionOfConstruction`: functional (787) · `currentUse`: 1_residential (761), 4_3_publicServices (15), 3_industrial (10), 4_2_retail (1) · `format`: jpeg (787) · `horizontalGeometryEstimatedAccuracy`: 0.1 (787) · `horizontalGeometryReference`: footPrint (787)

**building_parts** (6355) — `heightBelowGround`: 3 (3527), 0 (2777), 6 (51) · `horizontalGeometryEstimatedAccuracy`: 0.1 (6355) · `horizontalGeometryReference`: footPrint (6355) · `namespace`: ES.SDGC.BU (6355) · `numberOfFloorsBelowGround`: 1 (3527), 0 (2777), 2 (51)

**other_constructions** (124) — `constructionNature`: openAirPool (124) · `namespace`: ES.SDGC.BU (124)

**addresses** (782) — `level`: siteLevel (744), 4 (1) · `namespace`: ES.SDGC.AD (744), ES.SDGC.AU (1) · `component`: #ES.SDGC.PD.28.005.28806 (528), #ES.SDGC.PD.28.005.28805 (215), #ES.SDGC.PD.28.005.0 (1) · `default`: true (744) · `method`: fromFeature (744)

**parcels** (826) — `namespace`: ES.SDGC.CP (826)

... (2 outputs left out)

,tipo,id,m²,partes,localId,namespace,specification,method
0,addresses,9831107VK6892N,0.0,1,28.005.1229.S-N.9831107VK6892N,ES.SDGC.AD,Parcel,fromFeature
1,addresses,9431804VK6893S,0.0,1,28.005.1345.30T.9431804VK6893S,ES.SDGC.AD,Entrance,fromFeature
2,addresses,9831101VK6893S,0.0,1,28.005.1345.34.9831101VK6893S,ES.SDGC.AD,Entrance,fromFeature
3,addresses,9432939VK6893S,0.0,1,28.005.1345.35.9432939VK6893S,ES.SDGC.AD,Entrance,fromFeature
4,addresses,9432938VK6893S,0.0,1,28.005.1345.35A.9432938VK6893S,ES.SDGC.AD,Entrance,fromFeature
5,addresses,9432937VK6893S,0.0,1,28.005.1345.35B.9432937VK6893S,ES.SDGC.AD,Entrance,fromFeature
6,addresses,9432936VK6893S,0.0,1,28.005.1345.35C.9432936VK6893S,ES.SDGC.AD,Entrance,fromFeature
7,addresses,9432935VK6893S,0.0,1,28.005.1345.35D.9432935VK6893S,ES.SDGC.AD,Entrance,fromFeature


,tipo,id,m²,partes,areaValue,beginLifespanVersion,localId
0,parcels,9937001VK6893N,407413.9,2,407414,2026-03-13T00:00:00,9937001VK6893N
1,parcels,9831101VK6893S,85192.0,2,85193,2019-11-28T00:00:00,9831101VK6893S
2,parcels,9937002VK7803S,63070.3,2,63070,2026-05-20T00:00:00,9937002VK7803S
3,parcels,0434126VK7803N,48312.4,2,48313,2026-05-20T00:00:00,0434126VK7803N
4,parcels,9831107VK6892N,44903.0,2,44903,2013-11-27T00:00:00,9831107VK6892N
5,parcels,9831103VK6892N,33286.2,2,33286,2024-03-26T00:00:00,9831103VK6892N
6,parcels,0439101VK7803N,27811.6,3,26565,2015-08-07T00:00:00,0439101VK7803N
7,parcels,9339301VK6893N,26605.0,2,26604,2010-01-20T00:00:00,9339301VK6893N


![Otros tipos de feature](out/figs/feature-types.png)

tipos de feature: contestaron 5 de 5 (buildings=787, building_parts=6355, other_constructions=124, addresses=782, parcels=826); 0 sin datos sobre el extent
